In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for all datetime conversions
tz = pytz.timezone('Asia/Shanghai')

# Load files into memory (store full dataframes for later steps)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamp columns to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
# Timestamps are in seconds (floored to minute)
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Total number of rows per file
metric_rows = metric_df.shape[0]
trace_rows = trace_df.shape[0]
log_rows = log_df.shape[0]
error_logs_rows = error_logs_df.shape[0]

# 2) Unique cmdb_id values with their row counts (top 50)
metric_cmdb_counts = metric_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
trace_cmdb_counts = trace_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
log_cmdb_counts = log_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)
error_cmdb_counts = error_logs_df['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(50)

# 3) metric.csv: unique (cmdb_id, kpi_name) pairs with counts and global P95 of 'value'; top 20 by p95 desc
# Ensure numeric and dropna for value quantile
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
metric_pairs = (metric_df
                .groupby(['cmdb_id', 'kpi_name'], as_index=False)
                .agg(count=('value', 'count'),
                     p95_value=('value', lambda s: float(s.dropna().quantile(0.95)) if s.dropna().shape[0] else np.nan))
               )
metric_pairs_top20 = metric_pairs.sort_values('p95_value', ascending=False).head(20)

# 4) trace.csv: unique (cmdb_id, trace_name) pairs with counts and global P95 of 'value'; top 20 by p95 desc
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
trace_pairs = (trace_df
               .groupby(['cmdb_id', 'trace_name'], as_index=False)
               .agg(count=('value', 'count'),
                    p95_value=('value', lambda s: float(s.dropna().quantile(0.95)) if s.dropna().shape[0] else np.nan))
              )
trace_pairs_top20 = trace_pairs.sort_values('p95_value', ascending=False).head(20)

# 5) log.csv: unique (cmdb_id, log_name) pairs with counts and global P95 of 'value'; top 20 by p95 desc
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')
log_pairs = (log_df
             .groupby(['cmdb_id', 'log_name'], as_index=False)
             .agg(count=('value', 'count'),
                  p95_value=('value', lambda s: float(s.dropna().quantile(0.95)) if s.dropna().shape[0] else np.nan))
            )
log_pairs_top20 = log_pairs.sort_values('p95_value', ascending=False).head(20)

# 6) error_logs.csv: list unique cmdb_id values with counts (top50) and up to 5 example timestamps and truncated messages (limit overall examples to 20 rows)
# Prepare truncated message and formatted timestamp in Asia/Shanghai
error_logs_df['message_trunc'] = error_logs_df['message'].astype(str).str.slice(0, 200)  # truncate to 200 chars
error_logs_df['timestamp_str_shanghai'] = error_logs_df['timestamp_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# For examples: take up to 5 per cmdb_id, but overall limit 20 rows
error_logs_examples = (error_logs_df
                       .sort_values(['cmdb_id', 'timestamp'])  # consistent ordering
                       .groupby('cmdb_id', group_keys=False)
                       .apply(lambda g: g.head(5))
                      ).reset_index(drop=True)
error_logs_examples = error_logs_examples[['cmdb_id', 'timestamp_str_shanghai', 'message_trunc']].head(20)

# Compact summaries prepared. Display them as variables (compact).
metric_rows, trace_rows, log_rows, error_logs_rows, \
metric_cmdb_counts, trace_cmdb_counts, log_cmdb_counts, error_cmdb_counts, \
metric_pairs_top20, trace_pairs_top20, log_pairs_top20, \
error_logs_examples

```
Out[1]:
```
import pandas as pd
# Build a compact plain-English summary using existing in-kernel variables and small extracts

# Totals
totals = f"Total rows — metric: {metric_rows}, trace: {trace_rows}, log: {log_rows}, error_logs: {error_logs_rows}"

# Top cmdb_id lists (compact)
def top_cmdb_list(df_counts, n=8):
    items = []
    for _, r in df_counts.head(n).iterrows():
        items.append(f"{r['cmdb_id']}:{int(r['count'])}")
    return ", ".join(items)

metric_top_cmdb = top_cmdb_list(metric_cmdb_counts, 8)
trace_top_cmdb = top_cmdb_list(trace_cmdb_counts, 8)
log_top_cmdb = top_cmdb_list(log_cmdb_counts, 8)
error_top_cmdb = top_cmdb_list(error_cmdb_counts, 8) if not error_cmdb_counts.empty else "none"

# Top metric (cmdb_id, kpi_name) by global P95 — show top 6 for brevity
def format_pairs(df_pairs, n=6, id_col='cmdb_id', name_col=None):
    lines = []
    for _, r in df_pairs.head(n).iterrows():
        p95 = r['p95_value']
        p95_str = "nan" if pd.isna(p95) else f"{p95:.3g}"
        name = r[name_col] if name_col else "<name>"
        lines.append(f"{r[id_col]} | {name} | count={int(r['count'])} | p95={p95_str}")
    return lines

metric_top_lines = format_pairs(metric_pairs_top20, 6, 'cmdb_id', 'kpi_name')
trace_top_lines = format_pairs(trace_pairs_top20, 10, 'cmdb_id', 'trace_name')
log_top_lines = format_pairs(log_pairs_top20, 10, 'cmdb_id', 'log_name')

# Error logs examples summary
if error_logs_rows == 0:
    error_summary = "No error_logs rows present."
else:
    # show up to 5 cmdb_id examples (already limited to 20 rows in error_logs_examples)
    ex_lines = []
    for _, r in error_logs_examples.iterrows():
        ex_lines.append(f"{r['cmdb_id']} @ {r['timestamp_str_shanghai']} | {r['message_trunc'][:120]!r}")
    error_summary = "Examples (" + str(len(error_logs_examples)) + "): " + " ; ".join(ex_lines)

# Compose final summary string
summary = (
    totals + "\n\n"
    "Top cmdbs (metric): " + metric_top_cmdb + "\n"
    "Top cmdbs (trace): " + trace_top_cmdb + "\n"
    "Top cmdbs (log): " + log_top_cmdb + "\n"
    "Top cmdbs (error_logs): " + error_top_cmdb + "\n\n"
    "Top metric (cmdb_id | kpi_name | count | p95) — top examples:\n  - " + "\n  - ".join(metric_top_lines) + "\n\n"
    "Top trace (cmdb_id | trace_name | count | p95) — top examples:\n  - " + "\n  - ".join(trace_top_lines) + "\n\n"
    "Top log (cmdb_id | log_name | count | p95) — top examples:\n  - " + "\n  - ".join(log_top_lines) + "\n\n"
    "Error logs: " + error_summary
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(29239, 6984, 389, 0,           cmdb_id  count
0         Mysql02   4655
1         Mysql01   4619
2        Tomcat02   2021
3        Tomcat03   2011
4        Tomcat04   2007
..            ...    ...
21   ServiceTest8    124
22   ServiceTest1    120
23  ServiceTest11    120
24   ServiceTest6    120
25       dockerA2     54

[26 rows x 2 columns],      cmdb_id  count
0       MG02   1197
1       MG01   1185
2   Tomcat02    465
3       IG02    462
4   dockerA2    462
5   Tomcat01    462
6   Tomcat03    462
7   Tomcat04    462
8       IG01    459
9   dockerB1    459
10  dockerB2    456
11  dockerA1    453,     cmdb_id  count
0  Tomcat04     67
1  Tomcat01     66
2  Tomcat02     66
3  Tomcat03     66
4  apache01     62
5  apache02     62, Empty DataFrame
Columns: [cmdb_id, count]
Index: [],        cmdb_id                                           kpi_name  count     p95_value
668    Mysql02               Mysql-MySQL_3306_Innodb data written     29  2.852537e+12
657    Mysql02  Mysql-MySQL_3306_Innodb buffer pool read requests     29  1.665892e+12
678    Mysql02             Mysql-MySQL_3306_Innodb os log written     29  7.557133e+11
444    Mysql01               Mysql-MySQL_3306_Innodb data written     29  6.463935e+11
708    Mysql02                         Mysql-MySQL_3306_Rows Read     29  5.445350e+11
639    Mysql02                 Mysql-MySQL_3306_Handler Read Prev     29  5.443390e+11
616    Mysql02                        Mysql-MySQL_3306_Bytes Sent     29  1.942485e+11
392    Mysql01                        Mysql-MySQL_3306_Bytes Sent     29  8.062484e+10
665    Mysql02                  Mysql-MySQL_3306_Innodb data read     29  3.317314e+10
454    Mysql01             Mysql-MySQL_3306_Innodb os log written     29  9.461396e+09
937    Redis01  redis-Redis_6379_Redis  (total_commands_proces...     20  7.123131e+09
1048   Redis02  redis-Redis_6379_Redis  (total_commands_proces...     20  6.352621e+09
925    Redis01            redis-Redis_6379_Redis  (keyspace_hits)     20  5.458982e+09
1036   Redis02            redis-Redis_6379_Redis  (keyspace_hits)     20  4.697598e+09
930    Redis01                redis-Redis_6379_Redis  (maxmemory)     20  4.294967e+09
1041   Redis02                redis-Redis_6379_Redis  (maxmemory)     20  4.294967e+09
1322  Tomcat02             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09
1444  Tomcat03             Tomcat-MEMORY_7441-MEMORY_JVMMaxMemory      6  4.187619e+09
1446  Tomcat03           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09
1205  Tomcat01           Tomcat-MEMORY_7441-MEMORY_JVMTotalMemory      6  4.187619e+09,       cmdb_id                     trace_name  count  p95_value
212  dockerB1           trace.self.row_count     31    1262.00
119  Tomcat01           trace.self.row_count     31     711.00
152  Tomcat03           trace.self.row_count     31     703.50
167  Tomcat04           trace.self.row_count     31     659.00
134  Tomcat02           trace.self.row_count     31     636.00
182  dockerA1           trace.self.row_count     31     612.00
197  dockerA2           trace.self.row_count     31     577.00
227  dockerB2           trace.self.row_count     31     467.50
17       IG02           trace.self.row_count     31     402.00
2        IG01           trace.self.row_count     31     398.00
56       MG01           trace.self.row_count     31     320.50
95       MG02           trace.self.row_count     31     308.50
50       MG01  trace.from_dockerB1.row_count     30      89.85
215  dockerB1        trace.to_MG01.row_count     30      89.85
218  dockerB1        trace.to_MG02.row_count     31      84.50
89       MG02  trace.from_dockerB1.row_count     31      84.50
104      MG02    trace.to_dockerB1.row_count     30      78.00
209  dockerB1      trace.from_MG02.row_count     30      78.00
206  dockerB1      trace.from_MG01.row_count     31      74.50
65       MG01    trace.to_dockerB1.row_count     31      74.50,      cmdb_id          log_name  count  p95_value
13  apache01     log.row_count     31   200.5000
15  apache02     log.row_count     31   195.0000
8   Tomcat03     log.row_count     31   106.0000
2   Tomcat01     log.row_count     31   104.0000
11  Tomcat04     log.row_count     31    98.5000
5   Tomcat02     log.row_count     31    94.5000
1   Tomcat01  log.gc_pause_max      4     0.1055
4   Tomcat02  log.gc_pause_max      4     0.0800
7   Tomcat03  log.gc_pause_max      4     0.0685
10  Tomcat04  log.gc_pause_max      5     0.0480
0   Tomcat01   log.error_count     31     0.0000
3   Tomcat02   log.error_count     31     0.0000
6   Tomcat03   log.error_count     31     0.0000
9   Tomcat04   log.error_count     31     0.0000
12  apache01   log.error_count     31     0.0000
14  apache02   log.error_count     31     0.0000, Empty DataFrame
Columns: [cmdb_id, timestamp_str_shanghai, message_trunc]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse previously loaded dataframes: metric_df, trace_df, log_df, error_logs_df
# timezone (UTC+8) already defined earlier as tz; if not, define it
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# Ensure metric_df exists
if 'metric_df' not in globals():
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) for allowed components using full series
metric_allowed_full = metric_df[metric_df['cmdb_id'].isin(allowed_components)].copy()
metric_thresholds = (
    metric_allowed_full
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(
        count_total_series='count',
        global_p95=lambda s: float(s.dropna().quantile(0.95)) if s.dropna().shape[0] else np.nan,
        global_p5 = lambda s: float(s.dropna().quantile(0.05)) if s.dropna().shape[0] else np.nan
    )
)

# 2) Filter metric.csv to incident window: March 25, 2021 23:00 to 23:30 (UTC+8)
start = pd.to_datetime('2021-03-25 23:00:00').tz_localize(tz)
end   = pd.to_datetime('2021-03-25 23:30:00').tz_localize(tz)

metric_window = metric_df[
    (metric_df['cmdb_id'].isin(allowed_components)) &
    (metric_df['timestamp_dt'] >= start) &
    (metric_df['timestamp_dt'] <= end)
].copy()

# Merge global thresholds into the windowed rows for per-row comparison
mw_with_thresh = metric_window.merge(
    metric_thresholds[['cmdb_id','kpi_name','global_p95','global_p5']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# 3) For each (cmdb_id,kpi_name) produce compact anomaly summary for the incident window
def group_anomaly_stats(g):
    total = int(g['value'].count())
    # comparisons: handle NaN thresholds gracefully (comparisons will be False)
    ge_count = int((g['value'] >= g['global_p95']).sum()) if 'global_p95' in g else 0
    le_count = int((g['value'] <= g['global_p5']).sum()) if 'global_p5' in g else 0
    # earliest anomaly timestamp (timestamp_dt) where condition holds
    cond = (g['value'] >= g['global_p95']) | (g['value'] <= g['global_p5'])
    if cond.any():
        earliest_ts = g.loc[cond, 'timestamp_dt'].min()
        earliest_ts_str = earliest_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
        earliest_ts_unix = int(earliest_ts.tz_convert(pytz.utc).timestamp())  # store unix sec if needed
    else:
        earliest_ts = pd.NaT
        earliest_ts_str = np.nan
        earliest_ts_unix = np.nan
    maxv = g['value'].max()
    minv = g['value'].min()
    # global thresholds come from merged columns (they are same across group)
    gp95 = float(g['global_p95'].iloc[0]) if pd.notna(g['global_p95'].iloc[0]) else np.nan
    gp5  = float(g['global_p5'].iloc[0]) if pd.notna(g['global_p5'].iloc[0]) else np.nan
    # severity ratio
    if (pd.isna(gp95)) or (gp95 == 0):
        severity = np.nan
    else:
        severity = float(maxv) / gp95 if pd.notna(maxv) else np.nan
    return pd.Series({
        'count_total_points_in_window': total,
        'count_points_ge_p95': ge_count,
        'count_points_le_p5': le_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'earliest_anomaly_timestamp_shanghai': earliest_ts_str,
        'max_value_in_window': maxv,
        'min_value_in_window': minv,
        'global_p95': gp95,
        'global_p5': gp5,
        'severity_ratio': severity
    })

anomaly_stats = (
    mw_with_thresh
    .groupby(['cmdb_id','kpi_name'], sort=False)
    .apply(group_anomaly_stats)
    .reset_index()
)

# 4) Return only rows where count_points_ge_p95>0 OR count_points_le_p5>0.
anomaly_filtered = anomaly_stats[
    (anomaly_stats['count_points_ge_p95'] > 0) |
    (anomaly_stats['count_points_le_p5'] > 0)
].copy()

# 5) Sort by severity_ratio descending and return top 20 rows.
anomaly_filtered['severity_ratio'] = anomaly_filtered['severity_ratio'].replace([np.inf, -np.inf], np.nan)
anomaly_results_top20 = anomaly_filtered.sort_values('severity_ratio', ascending=False).head(20)

# Select and order the required columns compactly
display_cols = [
    'cmdb_id','kpi_name','global_p95','global_p5',
    'count_total_points_in_window','count_points_ge_p95','count_points_le_p5',
    'earliest_anomaly_timestamp','earliest_anomaly_timestamp_shanghai',
    'max_value_in_window','min_value_in_window','severity_ratio'
]
anomaly_results_top20 = anomaly_results_top20[display_cols]

# 6) Store intermediate variables in kernel for later use: metric_thresholds, metric_window (already defined)
# (They are available as metric_thresholds and metric_window variables)

# Final compact output: top 20 anomaly rows
anomaly_results_top20

```
Out[2]:
```
import pandas as pd

# Build a concise plain-English summary using existing in-kernel variables
# (expects anomaly_filtered, anomaly_results_top20, metric_thresholds, metric_window to exist)

total_anomalies = int(anomaly_filtered.shape[0]) if 'anomaly_filtered' in globals() else 0
window_start = "2021-03-25 23:00:00 (UTC+8)"
window_end = "2021-03-25 23:30:00 (UTC+8)"
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

top_df = anomaly_results_top20 if 'anomaly_results_top20' in globals() else pd.DataFrame()
top_n = min(10, top_df.shape[0])

lines = []
lines.append(f"Anomaly detection summary for metrics (allowed components: {', '.join(allowed)})")
lines.append(f"Incident window: {window_start} to {window_end}")
lines.append(f"Total (cmdb_id, kpi_name) pairs with anomalies in window: {total_anomalies}")
lines.append("")
if top_n == 0:
    lines.append("No anomalies detected in the specified window for allowed components.")
else:
    lines.append(f"Top {top_n} anomalies sorted by severity_ratio (descending):")
    for i in range(top_n):
        r = top_df.iloc[i]
        sev = r['severity_ratio']
        sev_str = "nan" if pd.isna(sev) else f"{sev:.3f}"
        lines.append(
            f"{i+1}. {r['cmdb_id']} | {r['kpi_name']} | severity={sev_str} | "
            f"count_in_window={int(r['count_total_points_in_window'])} | "
            f">=p95={int(r['count_points_ge_p95'])} | <=p5={int(r['count_points_le_p5'])} | "
            f"earliest_anomaly={r['earliest_anomaly_timestamp_shanghai']}"
        )

lines.append("")
lines.append("Notes:")
lines.append("- Global P95/P5 thresholds for all allowed (cmdb_id, kpi_name) pairs are stored in variable: metric_thresholds")
lines.append("- The filtered metric rows for the incident window are stored in variable: metric_window")
lines.append("- severity_ratio = max_value_in_window / global_p95 (NaN if global_p95 is 0 or NaN)")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         kpi_name  global_p95  global_p5  count_total_points_in_window  count_points_ge_p95  count_points_le_p5 earliest_anomaly_timestamp earliest_anomaly_timestamp_shanghai  max_value_in_window  min_value_in_window  severity_ratio
175    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.100000   0.000000                            19                    1                  18  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             1.000000             0.000000       10.000000
225   Tomcat01                           OSLinux-CPU_CPU_CPUWio    0.278558   0.014537                            31                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             1.554800             0.012470        5.581602
310   Tomcat03     OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    0.500000   0.000000                            31                    2                  29  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             2.000000             0.000000        4.000000
256   Tomcat02                           OSLinux-CPU_CPU_CPUWio    0.304274   0.020783                            29                    2                   3  2021-03-25 23:03:00+08:00             2021-03-25 23:03:00 CST             1.139100             0.020782        3.743665
1085  Tomcat02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.065000   0.000000                            15                    1                  14  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.216667             0.000000        3.333333
1017  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.300000   0.000000                            15                    1                  14  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             1.000000             0.000000        3.333333
1086  Tomcat02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    1.800000   0.000000                            15                    1                  14  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             6.000000             0.000000        3.333333
1025  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    0.300000   0.000000                            15                    1                  14  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             1.000000             0.000000        3.333333
1140  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.064155   0.000000                            14                    1                  13  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.183300             0.000000        2.857143
1141  Tomcat03  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    2.100000   0.000000                            14                    1                  13  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             6.000000             0.000000        2.857143
1083  Tomcat02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    4.600000   0.000000                            15                    1                  13  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST            13.000000             0.000000        2.826087
485       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead    2.266667   0.000000                            15                    1                  13  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             6.000000             0.000000        2.647059
991   Tomcat01                  OSLinux-CPU_CPU-2_SingleCpuidle   38.186470   0.000000                            15                    1                   9  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST            99.401400             0.000000        2.603053
13        IG01                           OSLinux-CPU_CPU_CPUWio    0.672765   0.016600                            18                    1                   5  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             1.729400             0.016600        2.570586
453   apache02                       OSLinux-CPU_CPU_CPUSysTime    0.598095   0.202255                            18                    1                   1  2021-03-25 23:06:00+08:00             2021-03-25 23:06:00 CST             1.491700             0.195200        2.494085
202    Redis02                       OSLinux-CPU_CPU_CPUSysTime    0.626413   0.362400                            31                    2                   2  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             1.524809             0.345400        2.434191
808    Mysql02                  Mysql-MySQL_3306_ThreadsRunning    4.600000   1.000000                            29                    2                  21  2021-03-25 23:02:00+08:00             2021-03-25 23:02:00 CST            11.000000             1.000000        2.391304
484       MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.078333   0.000000                            15                    1                  13  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.183333             0.000000        2.340431
454   apache02                      OSLinux-CPU_CPU_CPUUserTime    0.249075   0.084095                            18                    1                   1  2021-03-25 23:12:00+08:00             2021-03-25 23:12:00 CST             0.577600             0.066500        2.318980
143    Mysql01                       OSLinux-CPU_CPU_CPUSysTime    0.679680   0.295760                            17                    1                   1  2021-03-25 23:09:00+08:00             2021-03-25 23:09:00 CST             1.570400             0.266000        2.310499```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone if present
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# allowed components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

# ensure trace_df is loaded and prepared
if 'trace_df' not in globals():
    trace_df = pd.read_csv('trace.csv')
if 'timestamp_dt' not in trace_df.columns:
    trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if trace_df['value'].dtype.kind not in 'fi':
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) for allowed components using full series
trace_allowed_full = trace_df[trace_df['cmdb_id'].isin(allowed_components)].copy()
trace_thresholds = (
    trace_allowed_full
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(
        count_total_series='count',
        global_p95=lambda s: float(s.dropna().quantile(0.95)) if s.dropna().shape[0] else np.nan,
        global_p5=lambda s: float(s.dropna().quantile(0.05)) if s.dropna().shape[0] else np.nan
    )
)

# 2) Filter trace rows to incident window: March 25, 2021 23:00 to 23:30 (UTC+8)
start = pd.to_datetime('2021-03-25 23:00:00').tz_localize(tz)
end   = pd.to_datetime('2021-03-25 23:30:00').tz_localize(tz)

trace_window = trace_df[
    (trace_df['cmdb_id'].isin(allowed_components)) &
    (trace_df['timestamp_dt'] >= start) &
    (trace_df['timestamp_dt'] <= end)
].copy()

# 3) Merge thresholds into the window rows for per-row comparisons
tw_with_thresh = trace_window.merge(
    trace_thresholds[['cmdb_id','trace_name','global_p95','global_p5']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# helper to compute per-group anomaly summary
def trace_group_stats(g):
    total = int(g['value'].count())
    gp95 = float(g['global_p95'].iloc[0]) if pd.notna(g['global_p95'].iloc[0]) else np.nan
    gp5  = float(g['global_p5'].iloc[0]) if pd.notna(g['global_p5'].iloc[0]) else np.nan
    ge_count = int((g['value'] >= gp95).sum()) if not pd.isna(gp95) else 0
    le_count = int((g['value'] <= gp5).sum()) if not pd.isna(gp5) else 0
    cond = (g['value'] >= gp95) | (g['value'] <= gp5)
    if cond.any():
        earliest_ts = g.loc[cond, 'timestamp_dt'].min()
        earliest_ts_str = earliest_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_ts = pd.NaT
        earliest_ts_str = np.nan
    maxv = g['value'].max()
    minv = g['value'].min()
    if (pd.isna(gp95)) or (gp95 == 0):
        severity = np.nan
    else:
        severity = float(maxv) / gp95 if pd.notna(maxv) else np.nan
    return pd.Series({
        'global_p95': gp95,
        'global_p5': gp5,
        'count_total_points_in_window': total,
        'count_points_ge_p95': ge_count,
        'count_points_le_p5': le_count,
        'earliest_anomaly_timestamp': earliest_ts,
        'earliest_anomaly_timestamp_shanghai': earliest_ts_str,
        'max_value_in_window': maxv,
        'min_value_in_window': minv,
        'severity_ratio': severity
    })

# 3-4) Group and filter anomalies
trace_anomaly_stats = (
    tw_with_thresh
    .groupby(['cmdb_id','trace_name'], sort=False)
    .apply(trace_group_stats)
    .reset_index()
)

trace_anomaly_filtered = trace_anomaly_stats[
    (trace_anomaly_stats['count_points_ge_p95'] > 0) |
    (trace_anomaly_stats['count_points_le_p5'] > 0)
].copy()

# 4-5) Sort by severity_ratio desc and limit to top 20
trace_anomaly_filtered['severity_ratio'] = trace_anomaly_filtered['severity_ratio'].replace([np.inf, -np.inf], np.nan)
trace_anomaly_top20 = trace_anomaly_filtered.sort_values('severity_ratio', ascending=False).head(20)

# Select required columns
display_cols = [
    'cmdb_id','trace_name','global_p95','global_p5',
    'count_total_points_in_window','count_points_ge_p95','count_points_le_p5',
    'earliest_anomaly_timestamp','earliest_anomaly_timestamp_shanghai',
    'max_value_in_window','min_value_in_window','severity_ratio'
]
trace_anomaly_top20 = trace_anomaly_top20[display_cols]

# Output compact results and keep trace_thresholds and trace_window in kernel for follow-up
trace_thresholds, trace_window, trace_anomaly_top20

```
Out[3]:
```
# Build a concise plain-English summary of the trace-level anomaly results using in-kernel variables
import pandas as pd
import numpy as np

# Expecting these variables in kernel: trace_thresholds, trace_window, trace_anomaly_top20
# Fallback checks
_tt = trace_thresholds if 'trace_thresholds' in globals() else pd.DataFrame()
_tw = trace_window if 'trace_window' in globals() else pd.DataFrame()
_ttop = trace_anomaly_top20 if 'trace_anomaly_top20' in globals() else pd.DataFrame()

total_pairs = int(_tt.shape[0])
window_rows = int(_tw.shape[0])
anomaly_pairs = int((_ttop.shape[0]) if not _ttop.empty else 0)

lines = []
lines.append(f"Trace anomaly summary for allowed components (window 2021-03-25 23:00–23:30 UTC+8)")
lines.append(f"Global trace pairs analyzed: {total_pairs}")
lines.append(f"Trace rows in window: {window_rows}")
lines.append(f"Pairs with anomalies in window (>=P95 or <=P5): {anomaly_pairs}")
lines.append("")

# Show top up to 10 anomalies sorted by severity (from trace_anomaly_top20)
top_n = min(10, anomaly_pairs)
if top_n == 0:
    lines.append("No trace anomalies detected in the specified window for allowed components.")
else:
    lines.append(f"Top {top_n} trace anomalies (severity_ratio desc):")
    for i in range(top_n):
        r = _ttop.iloc[i]
        sev = r['severity_ratio']
        sev_str = "nan" if pd.isna(sev) else f"{sev:.3f}"
        lines.append(
            f"{i+1}. {r['cmdb_id']} | {r['trace_name']} | severity={sev_str} | "
            f"count_in_window={int(r['count_total_points_in_window'])} | "
            f">=p95={int(r['count_points_ge_p95'])} | <=p5={int(r['count_points_le_p5'])} | "
            f"earliest={r['earliest_anomaly_timestamp_shanghai']}"
        )

lines.append("")
lines.append("Stored variables for follow-up:")
lines.append("- trace_thresholds : global P95/P5 per (cmdb_id, trace_name)")
lines.append("- trace_window     : trace rows filtered to the incident window")
lines.append("- trace_anomaly_top20 : top anomaly summary rows shown above")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                       trace_name  count_total_series  global_p95   global_p5
0        IG01         trace.self.duration_mean                  31    0.333788    0.051533
1        IG01          trace.self.duration_p95                  31    1.228475    0.120050
2        IG01             trace.self.row_count                  31  398.000000  277.000000
3        IG01  trace.to_Tomcat01.duration_mean                  31    0.182202    0.034932
4        IG01   trace.to_Tomcat01.duration_p95                  31    1.010500    0.046550
..        ...                              ...                 ...         ...         ...
169  Tomcat04       trace.to_MG01.duration_p95                  31    0.812150    0.031850
170  Tomcat04          trace.to_MG01.row_count                  31   51.500000   26.000000
171  Tomcat04      trace.to_MG02.duration_mean                  31    0.101140    0.018831
172  Tomcat04       trace.to_MG02.duration_p95                  31    0.657350    0.037650
173  Tomcat04          trace.to_MG02.row_count                  31   53.500000   30.500000

[174 rows x 5 columns],        timestamp   cmdb_id                       trace_name       value              timestamp_dt
0     1616684400      IG01         trace.self.duration_mean    0.404189 2021-03-25 23:00:00+08:00
1     1616684400      IG01          trace.self.duration_p95    1.388650 2021-03-25 23:00:00+08:00
2     1616684400      IG01             trace.self.row_count  344.000000 2021-03-25 23:00:00+08:00
3     1616684400      IG01  trace.to_Tomcat01.duration_mean    0.036205 2021-03-25 23:00:00+08:00
4     1616684400      IG01   trace.to_Tomcat01.duration_p95    0.053800 2021-03-25 23:00:00+08:00
...          ...       ...                              ...         ...                       ...
6946  1616686200  Tomcat04       trace.to_MG01.duration_p95    0.023000 2021-03-25 23:30:00+08:00
6947  1616686200  Tomcat04          trace.to_MG01.row_count    1.000000 2021-03-25 23:30:00+08:00
6948  1616686200  Tomcat04      trace.to_MG02.duration_mean    0.025000 2021-03-25 23:30:00+08:00
6949  1616686200  Tomcat04       trace.to_MG02.duration_p95    0.025000 2021-03-25 23:30:00+08:00
6950  1616686200  Tomcat04          trace.to_MG02.row_count    1.000000 2021-03-25 23:30:00+08:00

[5154 rows x 5 columns],       cmdb_id                         trace_name  global_p95  global_p5  count_total_points_in_window  count_points_ge_p95  count_points_le_p5 earliest_anomaly_timestamp earliest_anomaly_timestamp_shanghai  max_value_in_window  min_value_in_window  severity_ratio
133  Tomcat02            trace.self.duration_p95    0.031525   0.020000                            31                    2                   2  2021-03-25 23:11:00+08:00             2021-03-25 23:11:00 CST             0.367000             0.011000       11.641554
52       MG01   trace.from_dockerB2.duration_p95    0.149805   0.007560                            30                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             1.076350             0.006900        7.185007
51       MG01  trace.from_dockerB2.duration_mean    0.040964   0.006267                            30                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.213250             0.006000        5.205822
33       MG01  trace.from_Tomcat02.duration_mean    0.146740   0.016332                            31                    2                   2  2021-03-25 23:16:00+08:00             2021-03-25 23:16:00 CST             0.725000             0.012956        4.940695
135  Tomcat02        trace.to_MG01.duration_mean    0.146740   0.016332                            31                    2                   2  2021-03-25 23:16:00+08:00             2021-03-25 23:16:00 CST             0.725000             0.012956        4.940695
54       MG01           trace.self.duration_mean    0.064511   0.023752                            31                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.256667             0.020524        3.978624
91       MG02   trace.from_dockerB2.duration_p95    0.092525   0.007735                            30                    2                   2  2021-03-25 23:10:00+08:00             2021-03-25 23:10:00 CST             0.367400             0.006700        3.970819
126  Tomcat02      trace.from_IG01.duration_mean    0.193097   0.027328                            31                    2                   2  2021-03-25 23:11:00+08:00             2021-03-25 23:11:00 CST             0.738000             0.019929        3.821923
6        IG01    trace.to_Tomcat02.duration_mean    0.193097   0.027328                            31                    2                   2  2021-03-25 23:11:00+08:00             2021-03-25 23:11:00 CST             0.738000             0.019929        3.821923
132  Tomcat02           trace.self.duration_mean    0.019921   0.004826                            31                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.066727             0.002023        3.349581
88       MG02   trace.from_dockerB1.duration_p95    0.027050   0.009025                            31                    2                   2  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             0.076750             0.009000        2.837338
55       MG01            trace.self.duration_p95    0.263400   0.047125                            31                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.654800             0.036600        2.485953
87       MG02  trace.from_dockerB1.duration_mean    0.019541   0.007173                            31                    2                   2  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             0.039694             0.007100        2.031319
81       MG02  trace.from_dockerA1.duration_mean    0.083487   0.006388                            30                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.168400             0.006143        2.017068
9        IG01    trace.to_Tomcat03.duration_mean    0.164165   0.025592                            30                    2                   2  2021-03-25 23:02:00+08:00             2021-03-25 23:02:00 CST             0.317229             0.021000        1.932375
141  Tomcat03      trace.from_IG01.duration_mean    0.164165   0.025592                            30                    2                   2  2021-03-25 23:02:00+08:00             2021-03-25 23:02:00 CST             0.317229             0.021000        1.932375
63       MG01    trace.to_dockerB1.duration_mean    0.193263   0.020025                            31                    2                   2  2021-03-25 23:01:00+08:00             2021-03-25 23:01:00 CST             0.372000             0.009500        1.924838
15       IG02           trace.self.duration_mean    0.234545   0.053264                            31                    2                   2  2021-03-25 23:00:00+08:00             2021-03-25 23:00:00 CST             0.434644             0.030750        1.853133
58       MG01     trace.to_dockerA1.duration_p95    0.261245   0.028708                            30                    2                   2  2021-03-25 23:02:00+08:00             2021-03-25 23:02:00 CST             0.465700             0.028450        1.782618
49       MG01   trace.from_dockerB1.duration_p95    0.063250   0.009000                            30                    2                   5  2021-03-25 23:05:00+08:00             2021-03-25 23:05:00 CST             0.109850             0.009000        1.736759)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import re
import pytz

# reuse timezone
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Required in-kernel variables: anomaly_filtered, trace_anomaly_top20, trace_window
# Ensure they exist
for var in ('anomaly_filtered', 'trace_anomaly_top20', 'trace_window'):
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel.")

# Work on a copy to be safe
af = anomaly_filtered.copy()
tat = trace_anomaly_top20.copy()
tw = trace_window.copy()

# Ensure timestamp columns are datetime tz-aware; anomaly_filtered earliest_anomaly_timestamp may be tz-aware
# If it's object, try to convert
if af['earliest_anomaly_timestamp'].dtype == object:
    af['earliest_anomaly_timestamp'] = pd.to_datetime(af['earliest_anomaly_timestamp'])

if 'earliest_anomaly_timestamp' in tat.columns and tat['earliest_anomaly_timestamp'].dtype == object:
    tat['earliest_anomaly_timestamp'] = pd.to_datetime(tat['earliest_anomaly_timestamp'])

# Prepare per-cmdb aggregation
cmdbs = af['cmdb_id'].unique()

rows = []
to_re = re.compile(r'\bto_([A-Za-z0-9]+)')
from_re = re.compile(r'\bfrom_([A-Za-z0-9]+)')

for cm in cmdbs:
    sub = af[af['cmdb_id'] == cm].copy()
    if sub.shape[0] == 0:
        continue
    # earliest metric anomaly timestamp (min of per-pair earliest)
    earliest_ts = sub['earliest_anomaly_timestamp'].min()
    if pd.isna(earliest_ts):
        earliest_ts_str = np.nan
    else:
        # ensure tz and format in Asia/Shanghai
        try:
            earliest_ts_sh = earliest_ts.tz_convert(tz)
        except Exception:
            earliest_ts_sh = earliest_ts.tz_localize(tz)
        earliest_ts_str = earliest_ts_sh.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_max_sev = sub['severity_ratio'].max(skipna=True)
    count_metric_pairs = int(sub.shape[0])
    # top metric kpis: sort by severity desc then earliest time asc
    sub['sev_sort'] = sub['severity_ratio'].fillna(-np.inf)
    sub_sorted = sub.sort_values(['sev_sort','earliest_anomaly_timestamp'], ascending=[False, True])
    topk = []
    for _, r in sub_sorted.head(5).iterrows():
        sev = r['severity_ratio']
        sev_str = "nan" if pd.isna(sev) else f"{sev:.3f}"
        et = r['earliest_anomaly_timestamp']
        if pd.isna(et):
            et_str = ""
        else:
            try:
                et_sh = et.tz_convert(tz)
            except Exception:
                et_sh = et.tz_localize(tz)
            et_str = et_sh.strftime('%Y-%m-%d %H:%M:%S %Z')
        topk.append(f"{r['kpi_name']}|{sev_str}|{et_str}")
    top_metric_kpis = ";".join(topk)

    # trace anomalies for this cmdb from trace_anomaly_top20
    tat_sub = tat[tat['cmdb_id'] == cm].copy() if not tat.empty else pd.DataFrame()
    if tat_sub.shape[0] == 0:
        earliest_trace_ts = np.nan
        earliest_trace_ts_str = np.nan
        top_trace_names_with_sev = np.nan
    else:
        earliest_trace_ts = tat_sub['earliest_anomaly_timestamp'].min()
        if pd.isna(earliest_trace_ts):
            earliest_trace_ts_str = np.nan
        else:
            try:
                et_sh = earliest_trace_ts.tz_convert(tz)
            except Exception:
                et_sh = earliest_trace_ts.tz_localize(tz)
            earliest_trace_ts_str = et_sh.strftime('%Y-%m-%d %H:%M:%S %Z')
        tat_sub['sev_sort'] = tat_sub['severity_ratio'].fillna(-np.inf)
        tat_sorted = tat_sub.sort_values(['sev_sort','earliest_anomaly_timestamp'], ascending=[False, True])
        tr_list = []
        for _, r in tat_sorted.head(5).iterrows():
            sev = r['severity_ratio']
            sev_str = "nan" if pd.isna(sev) else f"{sev:.3f}"
            et = r['earliest_anomaly_timestamp']
            if pd.isna(et):
                et_str = ""
            else:
                try:
                    et_sh = et.tz_convert(tz)
                except Exception:
                    et_sh = et.tz_localize(tz)
                et_str = et_sh.strftime('%Y-%m-%d %H:%M:%S %Z')
            tr_list.append(f"{r['trace_name']}|{sev_str}|{et_str}")
        top_trace_names_with_sev = ";".join(tr_list)

    # Inspect trace_window rows for interactions
    tw_sub = tw[tw['cmdb_id'] == cm].copy()
    # find 'to_' targets where this component is caller (trace_name contains to_<X>)
    to_targets = []
    from_callers = []
    if tw_sub.shape[0] > 0:
        # extract all to_ targets
        tw_sub['to_targets'] = tw_sub['trace_name'].apply(lambda s: to_re.findall(str(s)))
        tw_sub['from_sources'] = tw_sub['trace_name'].apply(lambda s: from_re.findall(str(s)))
        # explode lists
        to_exploded = tw_sub.explode('to_targets')
        from_exploded = tw_sub.explode('from_sources')
        # count targets
        to_counts = to_exploded['to_targets'].dropna()
        if not to_counts.empty:
            to_counts = to_counts.value_counts().head(5)
            to_targets = [f"{t}:{int(c)}" for t,c in zip(to_counts.index, to_counts.values)]
        # count callers (from_sources)
        from_counts = from_exploded['from_sources'].dropna()
        if not from_counts.empty:
            from_counts = from_counts.value_counts().head(5)
            from_callers = [f"{t}:{int(c)}" for t,c in zip(from_counts.index, from_counts.values)]

    to_targets_str = ",".join(to_targets) if to_targets else np.nan
    from_callers_str = ",".join(from_callers) if from_callers else np.nan

    rows.append({
        'cmdb_id': cm,
        'earliest_metric_anomaly_timestamp': earliest_ts,
        'earliest_metric_anomaly_timestamp_shanghai': earliest_ts_str,
        'metric_max_severity_ratio': metric_max_sev,
        'count_metric_anomalous_pairs': count_metric_pairs,
        'top_metric_kpis': top_metric_kpis,
        'earliest_trace_anomaly_timestamp': earliest_trace_ts,
        'earliest_trace_anomaly_timestamp_shanghai': earliest_trace_ts_str,
        'top_trace_names_with_sev': top_trace_names_with_sev,
        'to_targets': to_targets_str,
        'from_callers': from_callers_str
    })

component_metric_trace_summary = pd.DataFrame(rows)

# Sort by metric_max_severity_ratio desc and limit to top 20
component_metric_trace_summary = component_metric_trace_summary.sort_values('metric_max_severity_ratio', ascending=False).head(20)

# Keep output compact: show selected columns only
cols_to_show = [
    'cmdb_id','earliest_metric_anomaly_timestamp_shanghai','metric_max_severity_ratio',
    'count_metric_anomalous_pairs','top_metric_kpis',
    'earliest_trace_anomaly_timestamp_shanghai','top_trace_names_with_sev',
    'to_targets','from_callers'
]
component_metric_trace_summary = component_metric_trace_summary[cols_to_show]

# Display compact result (variable)
component_metric_trace_summary

```
Out[4]:
```
component_metric_trace_summary.shape[0], \
summary := (
    "Summary of component-level metric+trace correlations (top components sorted by metric severity):\n\n"
    f"Total components summarized: {component_metric_trace_summary.shape[0]}\n\n"
    "Top findings:\n"
    "- Mysql02: highest metric severity_ratio (10.0). Earliest metric anomaly at 2021-03-25 23:00:00 (UTC+8). "
    "Many metric KPIs flagged (224 pairs). No matching trace anomalies in the top trace set.\n"
    "- Tomcat01: severity ~5.58, earliest metric anomaly 2021-03-25 23:01:00. Counted 120 anomalous metric pairs. "
    "Top metric KPI: OSLinux-CPU_CPU_CPUWio. Trace anomalies present for Tomcat01 were not in top list, but trace-window interactions show Tomcat01 calls MG01 and MG02 frequently (MG01:93, MG02:90) and is called by IG01 and IG02 (IG01:93, IG02:90).\n"
    "- Tomcat03: severity 4.00, earliest metric anomaly 2021-03-25 23:00:00, 122 metric pairs flagged. There is an associated trace anomaly (earliest 2021-03-25 23:02:00) linking from IG01 -> Tomcat03. Common targets: MG02, MG01.\n"
    "- Tomcat02: severity ~3.74, earliest metric anomaly 23:03, 122 metric pairs flagged. Associated trace anomalies exist (e.g., trace.self.duration_p95) with earliest trace anomaly 23:01. Tomcat02 frequently calls MG01 and MG02 and is called by IG01/IG02.\n"
    "- MG01 and MG02: both show disk-related metric anomalies (DSKRead/DSKBps) with moderate severity (MG01 ~2.65, MG02 ~2.27). Their traces show many interactions with dockerA/B and Tomcat nodes (e.g., MG01 called by Tomcat01/02/04/03; MG01 calls dockerA2/dockerB1/dockerA1/dockerB2 in the window).\n"
    "- IG01 and IG02: CPU-related metric anomalies with earliest at 23:00. IG01 has trace anomalies pointing to Tomcat02 (IG01 -> Tomcat02). IG02 also shows trace.self.duration_mean anomalies.\n"
    "- Several components (Mysql01, Redis01/02, apache01/02, Tomcat04) also have metric anomalies but either lack top-trace anomalies or have smaller severity ratios.\n\n"
    "Interaction summary (from trace window parsing):\n"
    "- Tomcat nodes commonly call MG01 and MG02 (counts ~90+ in window).\n"
    "- MG nodes show many calls to docker* targets and receive calls from Tomcat nodes.\n"
    "- IG nodes appear as callers to Tomcat nodes (e.g., IG01 -> Tomcat02) in the window.\n\n"
    "Stored variables for follow-up:\n"
    "- component_metric_trace_summary : per-component compact summary (this table)\n"
    "- metric_thresholds, metric_window : per-(cmdb_id,kpi) global thresholds and filtered metric rows\n"
    "- trace_thresholds, trace_window : per-(cmdb_id,trace_name) global thresholds and filtered trace rows\n\n"
    "Recommendation: prioritize investigation on Mysql02 (highest metric severity), Tomcat01/02/03 (high severity + trace interactions with MG and IG), and MG01/MG02 (disk anomalies and heavy interaction traffic)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_metric_anomaly_timestamp_shanghai  metric_max_severity_ratio  count_metric_anomalous_pairs                                    top_metric_kpis earliest_trace_anomaly_timestamp_shanghai                           top_trace_names_with_sev                                       to_targets                                       from_callers
5    Mysql02                    2021-03-25 23:00:00 CST                  10.000000                           224  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT|1...                                       NaN                                                NaN                                              NaN                                                NaN
8   Tomcat01                    2021-03-25 23:00:00 CST                   5.581602                           120  OSLinux-CPU_CPU_CPUWio|5.582|2021-03-25 23:01:...                                       NaN                                                NaN                                  MG01:93,MG02:90                             IG01:93,IG02:90,root:3
10  Tomcat03                    2021-03-25 23:00:00 CST                   4.000000                           122  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT|4...                   2021-03-25 23:02:00 CST  trace.from_IG01.duration_mean|1.932|2021-03-25...                                  MG02:93,MG01:90                             IG02:93,IG01:90,root:3
9   Tomcat02                    2021-03-25 23:00:00 CST                   3.743665                           122  OSLinux-CPU_CPU_CPUWio|3.744|2021-03-25 23:03:...                   2021-03-25 23:01:00 CST  trace.self.duration_p95|11.642|2021-03-25 23:1...                                  MG01:93,MG02:93                                    IG01:93,IG02:93
2       MG01                    2021-03-25 23:00:00 CST                   2.647059                           104  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...                   2021-03-25 23:01:00 CST  trace.from_dockerB2.duration_p95|7.185|2021-03...  dockerA2:93,dockerB1:93,dockerA1:90,dockerB2:90  Tomcat01:93,Tomcat02:93,Tomcat04:93,Tomcat03:9...
0       IG01                    2021-03-25 23:00:00 CST                   2.570586                            88  OSLinux-CPU_CPU_CPUWio|2.571|2021-03-25 23:00:...                   2021-03-25 23:02:00 CST  trace.to_Tomcat02.duration_mean|3.822|2021-03-...  Tomcat01:93,Tomcat02:93,Tomcat03:90,Tomcat04:90                                                NaN
13  apache02                    2021-03-25 23:00:00 CST                   2.494085                            81  OSLinux-CPU_CPU_CPUSysTime|2.494|2021-03-25 23...                                       NaN                                                NaN                                              NaN                                                NaN
7    Redis02                    2021-03-25 23:00:00 CST                   2.434191                           111  OSLinux-CPU_CPU_CPUSysTime|2.434|2021-03-25 23...                                       NaN                                                NaN                                              NaN                                                NaN
4    Mysql01                    2021-03-25 23:00:00 CST                   2.310499                           224  OSLinux-CPU_CPU_CPUSysTime|2.310|2021-03-25 23...                                       NaN                                                NaN                                              NaN                                                NaN
3       MG02                    2021-03-25 23:00:00 CST                   2.272727                           105  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps...                   2021-03-25 23:00:00 CST  trace.from_dockerB2.duration_p95|3.971|2021-03...  dockerA1:93,dockerA2:93,dockerB2:93,dockerB1:90  Tomcat02:93,Tomcat03:93,Tomcat04:93,dockerA2:9...
11  Tomcat04                    2021-03-25 23:00:00 CST                   2.000000                           123  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                                       NaN                                                NaN                                  MG01:93,MG02:93                                    IG02:93,IG01:90
6    Redis01                    2021-03-25 23:00:00 CST                   1.996971                           115  OSLinux-CPU_CPU_CPUWio|1.997|2021-03-25 23:00:...                                       NaN                                                NaN                                              NaN                                                NaN
1       IG02                    2021-03-25 23:00:00 CST                   1.755207                            88  OSLinux-CPU_CPU-0_SingleCpuUtil|1.755|2021-03-...                   2021-03-25 23:00:00 CST  trace.self.duration_mean|1.853|2021-03-25 23:0...  Tomcat02:93,Tomcat03:93,Tomcat04:93,Tomcat01:90                                                NaN
12  apache01                    2021-03-25 23:00:00 CST                   1.666667                            81  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...                                       NaN                                                NaN                                              NaN                                                NaN```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse in-kernel variables: metric_thresholds, metric_window, trace_window, anomaly_filtered, component_metric_trace_summary
# Candidate components list
candidates = ['Mysql02','Tomcat01','Tomcat02','Tomcat03','Tomcat04','MG01','MG02','IG01','IG02','Redis01','Redis02','apache01','apache02']

# Ensure required variables exist
for v in ('metric_thresholds','metric_window','trace_window','anomaly_filtered'):
    if v not in globals():
        raise NameError(f"Required variable '{v}' not found in kernel.")

# timezone
tz = pytz.timezone('Asia/Shanghai')

# Prepare merged metric window with thresholds
mw = metric_window.merge(
    metric_thresholds[['cmdb_id','kpi_name','global_p95','global_p5']],
    on=['cmdb_id','kpi_name'],
    how='left'
).copy()

# Work on only pairs that were flagged as anomalous (use anomaly_filtered pairs) and within candidate components
pairs_flagged = anomaly_filtered[
    anomaly_filtered['cmdb_id'].isin(candidates)
][['cmdb_id','kpi_name']].drop_duplicates()

# Helper to compute earliest breach and consecutive run for a given group's rows
def compute_earliest_breach_run(df_pair):
    # df_pair expected sorted by timestamp_dt asc
    df_pair = df_pair.sort_values('timestamp_dt').reset_index(drop=True)
    gp95 = float(df_pair['global_p95'].iloc[0]) if pd.notna(df_pair['global_p95'].iloc[0]) else np.nan
    gp5  = float(df_pair['global_p5'].iloc[0]) if pd.notna(df_pair['global_p5'].iloc[0]) else np.nan
    # Boolean masks
    high_mask = df_pair['value'] >= gp95 if not pd.isna(gp95) else pd.Series([False]*len(df_pair))
    low_mask  = df_pair['value'] <= gp5  if not pd.isna(gp5) else pd.Series([False]*len(df_pair))
    breach_mask = high_mask | low_mask
    if not breach_mask.any():
        return None  # no breach in window
    # earliest breach index
    idx = int(breach_mask.idxmax())
    # determine breach type: prefer high if both True
    if high_mask.iloc[idx]:
        breach_type = "high"
        cond_mask = high_mask
    else:
        breach_type = "low"
        cond_mask = low_mask
    # earliest timestamp (tz-aware)
    earliest_ts = df_pair.loc[idx, 'timestamp_dt']
    earliest_ts_epoch = int(earliest_ts.tz_convert(pytz.utc).timestamp())
    earliest_ts_sh = earliest_ts.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    # Now compute consecutive run starting at idx
    run_vals = []
    run_idxs = []
    prev_ts = earliest_ts
    n = len(df_pair)
    for j in range(idx, n):
        row_ts = df_pair.loc[j, 'timestamp_dt']
        # require consecutive minute: difference from previous is 60s for j>idx
        if j == idx:
            ok_time = True
        else:
            diff = (row_ts - prev_ts).total_seconds()
            ok_time = (diff == 60)
        if not ok_time:
            break
        # Check same breach condition for this row
        if not cond_mask.iloc[j]:
            break
        run_idxs.append(j)
        run_vals.append(df_pair.loc[j, 'value'])
        prev_ts = row_ts
    run_length = len(run_idxs)  # minutes
    if run_length == 0:
        return None
    max_in_run = float(np.nanmax(run_vals)) if len(run_vals)>0 else np.nan
    min_in_run = float(np.nanmin(run_vals)) if len(run_vals)>0 else np.nan
    return {
        'earliest_anom_timestamp_epoch': earliest_ts_epoch,
        'earliest_anom_timestamp_shanghai': earliest_ts_sh,
        'breach_type': breach_type,
        'run_length_minutes': run_length,
        'max_value_in_run': max_in_run,
        'min_value_in_run': min_in_run,
        'global_p95': gp95,
        'global_p5': gp5
    }

# 1) Compute per-(cmdb_id,kpi_name) earliest breach/run for flagged pairs
pair_records = []
for _, row in pairs_flagged.iterrows():
    cm = row['cmdb_id']
    kpi = row['kpi_name']
    df_pair = mw[(mw['cmdb_id']==cm) & (mw['kpi_name']==kpi)].copy()
    if df_pair.empty:
        continue
    rec = compute_earliest_breach_run(df_pair)
    if rec is None:
        continue
    rec.update({'cmdb_id': cm, 'kpi_name': kpi})
    pair_records.append(rec)

pairs_earliest_df = pd.DataFrame(pair_records)
# keep one row per pair
if not pairs_earliest_df.empty:
    # reorder columns
    pairs_earliest_df = pairs_earliest_df[[
        'cmdb_id','kpi_name','breach_type','earliest_anom_timestamp_epoch','earliest_anom_timestamp_shanghai',
        'run_length_minutes','max_value_in_run','min_value_in_run','global_p95','global_p5'
    ]]

# 2) For each component keep only earliest per-component anomaly row
comp_rows = []
for cm in candidates:
    sub = pairs_earliest_df[pairs_earliest_df['cmdb_id']==cm].copy() if not pairs_earliest_df.empty else pd.DataFrame()
    if sub.empty:
        continue
    # earliest by epoch
    sub_sorted = sub.sort_values('earliest_anom_timestamp_epoch', ascending=True)
    r = sub_sorted.iloc[0]
    breach = r['breach_type']
    gp95 = r['global_p95']
    gp5 = r['global_p5']
    maxv = r['max_value_in_run']
    minv = r['min_value_in_run']
    # metric_severity_ratio: for high use max_in_run / global_p95; for low use min_in_run / global_p5
    if breach == 'high':
        if pd.isna(gp95) or gp95 == 0:
            severity = np.nan
        else:
            severity = maxv / gp95
    else:
        if pd.isna(gp5) or gp5 == 0:
            severity = np.nan
        else:
            severity = minv / gp5
    comp_rows.append({
        'cmdb_id': cm,
        'earliest_kpi': r['kpi_name'],
        'breach_type': breach,
        'earliest_anom_timestamp_epoch': int(r['earliest_anom_timestamp_epoch']),
        'earliest_anom_timestamp_shanghai': r['earliest_anom_timestamp_shanghai'],
        'run_length_minutes': int(r['run_length_minutes']),
        'max_value_in_run': float(maxv),
        'min_value_in_run': float(minv),
        'metric_severity_ratio': float(severity) if not pd.isna(severity) else np.nan
    })

component_earliest_faults = pd.DataFrame(comp_rows)

# 3) For each component's earliest_anom_timestamp retrieve trace_window rows at same timestamp involving component
# Build snapshots dict: cmdb -> DataFrame of up to 50 rows (selected columns)
snapshots = {}
if not component_earliest_faults.empty:
    # ensure trace_window timestamp_dt present and tz-aware
    tw = trace_window.copy()
    # standardize trace name string
    tw['trace_name'] = tw['trace_name'].astype(str)
    for _, r in component_earliest_faults.iterrows():
        cm = r['cmdb_id']
        epoch = int(r['earliest_anom_timestamp_epoch'])
        # convert epoch to tz-aware timestamp_dt
        ts = pd.to_datetime(epoch, unit='s', utc=True).tz_convert(tz)
        # find rows at same timestamp (exact equality)
        tw_at_ts = tw[tw['timestamp_dt'] == ts].copy()
        if tw_at_ts.empty:
            # also consider rows where trace_name contains to_<cm> or from_<cm> at that timestamp even if cmdb_id differs
            # but the above already includes rows with any cmdb_id; so if empty, then no rows
            snapshots[cm] = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','timestamp_dt'])
            continue
        # filter rows that involve the component: either cmdb_id == cm OR trace_name contains to_<cm> OR from_<cm>
        pattern_to = f"to_{cm}"
        pattern_from = f"from_{cm}"
        involved = tw_at_ts[
            (tw_at_ts['cmdb_id'] == cm) |
            (tw_at_ts['trace_name'].str.contains(pattern_to, na=False)) |
            (tw_at_ts['trace_name'].str.contains(pattern_from, na=False))
        ].copy()
        if involved.empty:
            snapshots[cm] = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','timestamp_dt'])
            continue
        # select columns: timestamp, cmdb_id, trace_name, value, timestamp_dt
        cols = ['timestamp','cmdb_id','trace_name','value','timestamp_dt']
        involved = involved[cols].head(50).reset_index(drop=True)
        snapshots[cm] = involved

# 4) Return compact result limited to candidate components, sorted by earliest_anom_timestamp ascending, top 20
if not component_earliest_faults.empty:
    component_earliest_faults = component_earliest_faults.sort_values('earliest_anom_timestamp_epoch', ascending=True).reset_index(drop=True)
    component_earliest_faults = component_earliest_faults.head(20)

# Store as requested
component_fault_trace_snapshots = snapshots

# Compact display variables
component_earliest_faults, list(component_fault_trace_snapshots.keys())[:20]

```
Out[5]:
```
component_earliest_faults_summary_lines = []
if 'component_earliest_faults' not in globals():
    component_earliest_faults_summary = "No component_earliest_faults found in kernel."
else:
    df = component_earliest_faults.copy()
    # ensure sorted earliest first
    if 'earliest_anom_timestamp_epoch' in df.columns:
        df = df.sort_values('earliest_anom_timestamp_epoch', ascending=True).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)
    # summarize up to 20 rows
    n = min(20, df.shape[0])
    component_earliest_faults_summary_lines.append(f"Component earliest faults summary (top {n} by time):")
    for i in range(n):
        r = df.iloc[i]
        cm = r['cmdb_id']
        kpi = r['earliest_kpi']
        breach = r['breach_type']
        ts_sh = r['earliest_anom_timestamp_shanghai']
        run_len = int(r['run_length_minutes'])
        maxv = r['max_value_in_run']
        minv = r['min_value_in_run']
        sev = r['metric_severity_ratio']
        sev_str = "nan" if pd.isna(sev) else f"{sev:.3f}"
        # trace snapshot info
        snap = component_fault_trace_snapshots.get(cm, None) if 'component_fault_trace_snapshots' in globals() else None
        if snap is None:
            snap_info = "no snapshot variable"
            snap_count = 0
        else:
            snap_count = 0 if snap.empty else snap.shape[0]
            snap_info = f"{snap_count} trace rows at earliest timestamp"
        component_earliest_faults_summary_lines.append(
            f"{i+1}. {cm} | KPI={kpi} | breach={breach} | earliest={ts_sh} | run_min={run_len} | "
            f"max_in_run={maxv:g} min_in_run={minv:g} | severity={sev_str} | trace_rows={snap_info}"
        )
    component_earliest_faults_summary_lines.append("")
    component_earliest_faults_summary_lines.append(
        "Key takeaways: Mysql02 shows highest-impact disk-related low breach at 2021-03-25 23:00 with many metric pairs flagged. "
        "Tomcat nodes (Tomcat01/02/03/04) show CPU-related high breaches around 23:00 with short runs. "
        "MG/IG nodes show JVM memory max breaches with long 31-minute runs (HeapMemoryMax) at 23:00. "
        "Redis and apache nodes also have early breaches at 23:00. "
        "Trace snapshots for each component (if any) are stored in component_fault_trace_snapshots; keys available: "
        + (", ".join(list(component_fault_trace_snapshots.keys())) if 'component_fault_trace_snapshots' in globals() else "none")
    )
    component_earliest_faults_summary = "\n".join(component_earliest_faults_summary_lines)

component_earliest_faults_summary, component_earliest_faults.shape[0] if 'component_earliest_faults' in globals() else 0

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                       earliest_kpi breach_type  earliest_anom_timestamp_epoch earliest_anom_timestamp_shanghai  run_length_minutes  max_value_in_run  min_value_in_run  metric_severity_ratio
0    Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...         low                     1616684400          2021-03-25 23:00:00 CST                   1      0.000000e+00      0.000000e+00                    NaN
1   Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil        high                     1616684400          2021-03-25 23:00:00 CST                   1      2.380200e+00      2.380200e+00               1.139451
2   Tomcat02                         OSLinux-CPU_CPU_CPUCpuUtil        high                     1616684400          2021-03-25 23:00:00 CST                   1      2.630620e+01      2.630620e+01               1.000006
3   Tomcat03                        OSLinux-CPU_CPU_CPUUserTime        high                     1616684400          2021-03-25 23:00:00 CST                   1      2.535240e+01      2.535240e+01               1.000596
4   Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuidle        high                     1616684400          2021-03-25 23:00:00 CST                   1      0.000000e+00      0.000000e+00                    NaN
5       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax        high                     1616684400          2021-03-25 23:00:00 CST                  31      2.040136e+09      2.040136e+09               1.000000
6       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax        high                     1616684400          2021-03-25 23:00:00 CST                  31      2.040136e+09      2.040136e+09               1.000000
7       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax        high                     1616684400          2021-03-25 23:00:00 CST                  31      2.040136e+09      2.040136e+09               1.000000
8       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax        high                     1616684400          2021-03-25 23:00:00 CST                  31      2.040136e+09      2.040136e+09               1.000000
9    Redis01                            OSLinux-CPU_CPU_CPULoad         low                     1616684400          2021-03-25 23:00:00 CST                   1      1.000000e+00      1.000000e+00               1.000000
10   Redis02                         OSLinux-CPU_CPU_CPUSysTime         low                     1616684400          2021-03-25 23:00:00 CST                   1      3.540000e-01      3.540000e-01               0.976821
11  apache01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        high                     1616684400          2021-03-25 23:00:00 CST                   2      7.800000e+01      7.800000e+01               1.000000
12  apache02         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity        high                     1616684400          2021-03-25 23:00:00 CST                   1      1.901450e+01      1.901450e+01               1.000000, ['Mysql02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'MG01', 'MG02', 'IG01', 'IG02', 'Redis01', 'Redis02', 'apache01', 'apache02'])```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing in-kernel variables: component_earliest_faults, metric_window, metric_thresholds, trace_window
# Ensure they exist
for v in ('component_earliest_faults','metric_window','metric_thresholds','trace_window'):
    if v not in globals():
        raise NameError(f"Required variable '{v}' not found in kernel.")

tz = pytz.timezone('Asia/Shanghai')

# Work on copies
cef = component_earliest_faults.copy()
mw = metric_window.copy()
mt = metric_thresholds.copy()
tw = trace_window.copy()

# Ensure timestamp_dt and timestamp columns exist and tz-aware
if 'timestamp_dt' not in mw.columns:
    mw['timestamp_dt'] = pd.to_datetime(mw['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'timestamp_dt' not in tw.columns:
    tw['timestamp_dt'] = pd.to_datetime(tw['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Build helper to get thresholds for a pair
def get_thresholds(cm, kpi):
    row = mt[(mt['cmdb_id']==cm) & (mt['kpi_name']==kpi)]
    if row.shape[0]==0:
        return np.nan, np.nan
    return float(row['global_p95'].iloc[0]), float(row['global_p5'].iloc[0])

# Prepare storage
run_rows = []
snapshots = {}

# For each component row in cef
for _, r in cef.iterrows():
    cm = r['cmdb_id']
    kpi = r['earliest_kpi']
    breach = r['breach_type']
    # start epoch from cef
    start_epoch = int(r['earliest_anom_timestamp_epoch'])
    start_ts = pd.to_datetime(start_epoch, unit='s', utc=True).tz_convert(tz)
    # get thresholds
    gp95, gp5 = get_thresholds(cm, kpi)
    # subset metric_window for this pair and sort
    dfp = mw[(mw['cmdb_id']==cm) & (mw['kpi_name']==kpi)].copy()
    if dfp.empty:
        # no data in window
        continue
    dfp = dfp.sort_values('timestamp_dt').reset_index(drop=True)
    # find start index matching start_ts (exact match)
    mask_start = dfp['timestamp_dt'] == start_ts
    if not mask_start.any():
        # If exact timestamp not found, try matching by epoch value in 'timestamp' column
        mask_start = dfp['timestamp'] == start_epoch
        if not mask_start.any():
            # cannot find start row; skip
            continue
    idx0 = int(np.flatnonzero(mask_start)[0])
    # build breach condition series based on breach type and thresholds
    if breach == 'high':
        if pd.isna(gp95):
            cond = pd.Series([False]*len(dfp))
        else:
            cond = dfp['value'] >= gp95
    else:
        if pd.isna(gp5):
            cond = pd.Series([False]*len(dfp))
        else:
            cond = dfp['value'] <= gp5
    # If start idx is not a breach (guard), find next breach index after idx0
    if not cond.iloc[idx0]:
        # find next True after idx0
        subsequent = cond.iloc[idx0:]
        if not subsequent.any():
            continue
        idx0 = idx0 + int(np.flatnonzero(subsequent)[0])
        start_ts = dfp.loc[idx0,'timestamp_dt']
        start_epoch = int(start_ts.tz_convert(pytz.utc).timestamp())
    # Now walk forward from idx0 while consecutive minutes and cond True
    run_idxs = []
    prev_ts = dfp.loc[idx0,'timestamp_dt']
    n = dfp.shape[0]
    for j in range(idx0, n):
        cur_ts = dfp.loc[j,'timestamp_dt']
        # check consecutive minute
        if j == idx0:
            ok_time = True
        else:
            diff = (cur_ts - prev_ts).total_seconds()
            ok_time = (diff == 60)
        if not ok_time:
            break
        if not cond.iloc[j]:
            break
        run_idxs.append(j)
        prev_ts = cur_ts
    if len(run_idxs) == 0:
        continue
    run_start_idx = run_idxs[0]
    run_end_idx = run_idxs[-1]
    run_start_ts = dfp.loc[run_start_idx,'timestamp_dt']
    run_end_ts = dfp.loc[run_end_idx,'timestamp_dt']
    run_start_epoch = int(run_start_ts.tz_convert(pytz.utc).timestamp())
    run_end_epoch = int(run_end_ts.tz_convert(pytz.utc).timestamp())
    run_length = len(run_idxs)
    run_values = dfp.loc[run_idxs,'value'].astype(float)
    if breach == 'high':
        extremal = float(run_values.max())
    else:
        extremal = float(run_values.min())
    # deviation_pct
    if breach == 'high':
        denom = gp95
        if pd.isna(denom) or denom == 0:
            deviation = np.nan
            sev_ratio = np.nan
        else:
            deviation = (extremal - denom) / denom
            sev_ratio = extremal / denom
    else:
        denom = gp5
        if pd.isna(denom) or denom == 0:
            deviation = np.nan
            sev_ratio = np.nan
        else:
            deviation = (denom - extremal) / denom
            sev_ratio = extremal / denom
    is_long = run_length >= 2
    candidate_true = bool(is_long or (pd.notna(deviation) and deviation >= 0.5))
    run_rows.append({
        'cmdb_id': cm,
        'earliest_kpi': kpi,
        'breach_type': breach,
        'run_start_epoch': run_start_epoch,
        'run_start_shanghai': run_start_ts.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'run_end_epoch': run_end_epoch,
        'run_end_shanghai': run_end_ts.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'run_length_minutes': int(run_length),
        'extremal_value_in_run': extremal,
        'global_p95': gp95,
        'global_p5': gp5,
        'deviation_pct': deviation,
        'metric_severity_ratio': sev_ratio,
        'is_long_run_bool': is_long,
        'candidate_true_fault': candidate_true
    })
    # build trace snapshot at run_start_epoch: exact timestamp rows involving component
    ts_exact = run_start_ts
    tw_at_ts = tw[tw['timestamp_dt'] == ts_exact].copy()
    if tw_at_ts.empty:
        snapshots[cm] = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','timestamp_dt'])
    else:
        pattern_to = f"to_{cm}"
        pattern_from = f"from_{cm}"
        involved = tw_at_ts[
            (tw_at_ts['cmdb_id'] == cm) |
            (tw_at_ts['trace_name'].str.contains(pattern_to, na=False)) |
            (tw_at_ts['trace_name'].str.contains(pattern_from, na=False))
        ].copy()
        snapshots[cm] = involved[['timestamp','cmdb_id','trace_name','value','timestamp_dt']].head(50).reset_index(drop=True)

# Build DataFrame
component_fault_runs = pd.DataFrame(run_rows)

# Sorting: candidate_true_fault True first, then deviation_pct desc (NaNs treated as -inf to push lower)
if not component_fault_runs.empty:
    component_fault_runs['deviation_sort'] = component_fault_runs['deviation_pct'].fillna(-np.inf)
    component_fault_runs = component_fault_runs.sort_values(['candidate_true_fault','deviation_sort'], ascending=[False, False]).drop(columns=['deviation_sort']).reset_index(drop=True)
    component_fault_runs = component_fault_runs.head(20)

# Store snapshots mapping in kernel variable (overwrite/assign)
component_fault_trace_snapshots = snapshots

# Final compact output: show top 20 summary rows (DataFrame variable)
component_fault_runs

```
Out[6]:
```
import pandas as pd

# Build a concise plain-English summary from in-kernel variables
# Expecting: component_fault_runs, component_fault_trace_snapshots
if 'component_fault_runs' not in globals():
    summary = "No component_fault_runs found in kernel."
else:
    df = component_fault_runs.copy()
    df_display = df[['cmdb_id','earliest_kpi','breach_type','run_length_minutes','extremal_value_in_run','global_p95','global_p5','deviation_pct','metric_severity_ratio','candidate_true_fault']].copy()
    total = df.shape[0]
    true_faults = df[df['candidate_true_fault'] == True]
    n_true = true_faults.shape[0]
    # prepare top true-faults details
    true_lines = []
    for _, r in true_faults.iterrows():
        true_lines.append(f"{r['cmdb_id']} ({r['earliest_kpi']}) run_min={int(r['run_length_minutes'])} breach={r['breach_type']} dev={r['deviation_pct'] if pd.notna(r['deviation_pct']) else 'nan'} sev={r['metric_severity_ratio'] if pd.notna(r['metric_severity_ratio']) else 'nan'}")
    # components with short/single-minute breaches (not candidate true)
    short_lines = []
    for _, r in df[df['candidate_true_fault'] == False].iterrows():
        short_lines.append(f"{r['cmdb_id']} ({r['earliest_kpi']}) run_min={int(r['run_length_minutes'])} breach={r['breach_type']} dev={r['deviation_pct'] if pd.notna(r['deviation_pct']) else 'nan'}")
    # snapshot keys
    snap_keys = sorted(list(component_fault_trace_snapshots.keys())) if 'component_fault_trace_snapshots' in globals() else []
    snap_count = len(snap_keys)
    snap_sample = ", ".join(snap_keys[:8]) + (", ..." if snap_count>8 else "")
    # Compose summary
    lines = []
    lines.append(f"Component fault run summary — {total} components analyzed (top 20 shown).")
    lines.append(f"Candidate true faults: {n_true} component(s).")
    if n_true:
        lines.append("High-priority (candidate true) components:")
        for L in true_lines:
            lines.append(" - " + L)
    else:
        lines.append("No candidate-true faults detected.")
    lines.append("")
    lines.append("Lower-priority / single-minute breaches (examples):")
    lines.extend(" - " + L for L in short_lines[:8])
    lines.append("")
    lines.append("Key observations:")
    lines.append(" - MG01, MG02, IG01, IG02: 31-minute sustained run on JVM HeapMemoryMax (extremal == global_p95, severity ~1.0).")
    lines.append(" - apache01: 2-minute sustained high memory usage (MEMUsedMemPerc), severity ~1.0.")
    lines.append(" - Tomcat nodes (Tomcat01/02/03/04) and Redis/Mysql/apache02 show single-minute breaches with small deviation or NaN severity — lower immediate priority.")
    lines.append("")
    lines.append(f"Trace snapshots captured for {snap_count} components (keys sample: {snap_sample}). Snapshots are stored in variable: component_fault_trace_snapshots.")
    lines.append("")
    lines.append("Recommendation: prioritize investigation in order — MG and IG JVM HeapMemoryMax issues first (sustained 31-min runs), then apache01 memory, then Tomcat nodes and storage/disk-related metrics as follow-up.")
    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       earliest_kpi breach_type  run_start_epoch       run_start_shanghai  run_end_epoch         run_end_shanghai  run_length_minutes  extremal_value_in_run    global_p95     global_p5  deviation_pct  metric_severity_ratio  is_long_run_bool  candidate_true_fault
0       MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax        high       1616684400  2021-03-25 23:00:00 CST     1616686200  2021-03-25 23:30:00 CST                  31           2.040136e+09  2.040136e+09  2.040136e+09       0.000000               1.000000              True                  True
1       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax        high       1616684400  2021-03-25 23:00:00 CST     1616686200  2021-03-25 23:30:00 CST                  31           2.040136e+09  2.040136e+09  2.040136e+09       0.000000               1.000000              True                  True
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax        high       1616684400  2021-03-25 23:00:00 CST     1616686200  2021-03-25 23:30:00 CST                  31           2.040136e+09  2.040136e+09  2.040136e+09       0.000000               1.000000              True                  True
3       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax        high       1616684400  2021-03-25 23:00:00 CST     1616686200  2021-03-25 23:30:00 CST                  31           2.040136e+09  2.040136e+09  2.040136e+09       0.000000               1.000000              True                  True
4   apache01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        high       1616684400  2021-03-25 23:00:00 CST     1616684460  2021-03-25 23:01:00 CST                   2           7.800000e+01  7.800000e+01  7.700000e+01       0.000000               1.000000              True                  True
5   Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil        high       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           2.380200e+00  2.088900e+00  1.114050e+00       0.139451               1.139451             False                 False
6    Redis02                         OSLinux-CPU_CPU_CPUSysTime         low       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           3.540000e-01  6.264130e-01  3.624000e-01       0.023179               0.976821             False                 False
7   Tomcat03                        OSLinux-CPU_CPU_CPUUserTime        high       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           2.535240e+01  2.533730e+01  2.521148e+01       0.000596               1.000596             False                 False
8   Tomcat02                         OSLinux-CPU_CPU_CPUCpuUtil        high       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           2.630620e+01  2.630605e+01  2.593025e+01       0.000006               1.000006             False                 False
9    Redis01                            OSLinux-CPU_CPU_CPULoad         low       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           1.000000e+00  1.086000e+00  1.000000e+00       0.000000               1.000000             False                 False
10  apache02         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity        high       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           1.901450e+01  1.901450e+01  1.901450e+01       0.000000               1.000000             False                 False
11   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...         low       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           0.000000e+00  1.680000e+00  0.000000e+00            NaN                    NaN             False                 False
12  Tomcat04                    OSLinux-CPU_CPU-1_SingleCpuidle        high       1616684400  2021-03-25 23:00:00 CST     1616684400  2021-03-25 23:00:00 CST                   1           0.000000e+00  0.000000e+00  0.000000e+00            NaN                    NaN             False                 False```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse in-kernel variables
for v in ('component_fault_runs','metric_window','metric_thresholds','trace_window'):
    if v not in globals():
        raise NameError(f"Required variable '{v}' not found in kernel.")

tz = pytz.timezone('Asia/Shanghai')

# 1) One-row summary per component where candidate_true_fault == True
cfr = component_fault_runs.copy()
candidates = cfr[cfr['candidate_true_fault'] == True].copy()
# select and order columns as requested
summary_cols = [
    'cmdb_id','run_start_epoch','run_start_shanghai','run_length_minutes',
    'earliest_kpi','breach_type','extremal_value_in_run','global_p95','global_p5',
    'deviation_pct','metric_severity_ratio','candidate_true_fault'
]
# Some column names may differ: ensure mapping
# If run_start_shanghai not present, check run_start_shanghai vs run_start_shanghai
if 'run_start_shanghai' not in candidates.columns and 'run_start_shanghai' in candidates.columns:
    pass
candidate_summary = candidates[summary_cols].copy().sort_values('run_start_epoch', ascending=True).reset_index(drop=True)

# 2) For each component retrieve trace_window rows at exact run_start_epoch and build compact aggregate
tw = trace_window.copy()
# ensure timestamp_dt present and tz-aware
if 'timestamp_dt' not in tw.columns:
    tw['timestamp_dt'] = pd.to_datetime(tw['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
tw['trace_name'] = tw['trace_name'].astype(str)

snapshot_map = {}  # store full DataFrames per component
agg_rows = []
for _, row in candidate_summary.iterrows():
    cm = row['cmdb_id']
    start_epoch = int(row['run_start_epoch'])
    run_start_ts = pd.to_datetime(start_epoch, unit='s', utc=True).tz_convert(tz)
    # rows at same timestamp
    tw_at = tw[tw['timestamp_dt'] == run_start_ts].copy()
    if tw_at.empty:
        involved = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','timestamp_dt'])
    else:
        pattern_to = f"to_{cm}"
        pattern_from = f"from_{cm}"
        involved = tw_at[
            (tw_at['cmdb_id'] == cm) |
            (tw_at['trace_name'].str.contains(pattern_to, na=False)) |
            (tw_at['trace_name'].str.contains(pattern_from, na=False))
        ].copy()
        involved = involved[['timestamp','cmdb_id','trace_name','value','timestamp_dt']].reset_index(drop=True)
    # store full snapshot in kernel mapping
    snapshot_map[cm] = involved
    # compact aggregate
    n_rows = int(involved.shape[0])
    if n_rows == 0:
        top_traces_str = np.nan
        is_downstream_callee = False
    else:
        top_counts = involved['trace_name'].value_counts().head(5)
        top_traces_str = ",".join([f"{name}:{int(cnt)}" for name,cnt in zip(top_counts.index, top_counts.values)])
        # downstream callee if any trace_name contains to_<cm> and the caller (cmdb_id) != cm
        mask_to = involved['trace_name'].str.contains(f"to_{cm}", na=False)
        if mask_to.any():
            # check if any of these rows have cmdb_id != cm (other caller)
            is_downstream_callee = bool(((involved[mask_to]['cmdb_id'] != cm)).any())
        else:
            is_downstream_callee = False
    agg_rows.append({
        'cmdb_id': cm,
        'run_start_epoch': start_epoch,
        'run_start_shanghai': run_start_ts.strftime('%Y-%m-%d %H:%M:%S %Z'),
        'number_of_rows_found': n_rows,
        'top_trace_name_counts': top_traces_str,
        'is_downstream_callee': is_downstream_callee
    })

trace_snapshot_aggregates = pd.DataFrame(agg_rows).sort_values('run_start_epoch', ascending=True).reset_index(drop=True)

# store full snapshots mapping in kernel variable (overwrite/update)
component_fault_trace_snapshots = snapshot_map

# 3) Mysql02 high-severity KPI anomalies inspection
mt = metric_thresholds.copy()
mw = metric_window.copy()
# focus on Mysql02 pairs present in metric_thresholds
mysql_pairs = mt[mt['cmdb_id']=='Mysql02'][['cmdb_id','kpi_name','global_p95','global_p5']].drop_duplicates()
mysql_records = []
# helper to find earliest breach and extremal in window
for _, p in mysql_pairs.iterrows():
    cm = p['cmdb_id']
    kpi = p['kpi_name']
    gp95 = float(p['global_p95']) if pd.notna(p['global_p95']) else np.nan
    gp5  = float(p['global_p5']) if pd.notna(p['global_p5']) else np.nan
    dfp = mw[(mw['cmdb_id']==cm) & (mw['kpi_name']==kpi)].copy()
    if dfp.empty:
        continue
    dfp = dfp.sort_values('timestamp_dt').reset_index(drop=True)
    # masks
    high_mask = dfp['value'] >= gp95 if not pd.isna(gp95) else pd.Series([False]*len(dfp))
    low_mask  = dfp['value'] <= gp5  if not pd.isna(gp5) else pd.Series([False]*len(dfp))
    breach_mask = high_mask | low_mask
    if not breach_mask.any():
        continue
    idx = int(breach_mask.idxmax())
    earliest_ts = dfp.loc[idx,'timestamp_dt']
    earliest_sh = earliest_ts.strftime('%Y-%m-%d %H:%M:%S %Z')
    # compute extremal in window (for high use max, for low use min)
    max_val = float(dfp['value'].max())
    min_val = float(dfp['value'].min())
    # determine severity: prefer high severity ratio (max/gp95) if gp95 valid, else for low use min/gp5
    sev = np.nan
    deviation = np.nan
    if high_mask.iloc[idx]:
        # high breach
        if (pd.notna(gp95) and gp95 != 0):
            sev = max_val / gp95
            deviation = (max_val - gp95) / gp95
    else:
        # low breach
        if (pd.notna(gp5) and gp5 != 0):
            sev = min_val / gp5
            deviation = (gp5 - min_val) / gp5
    mysql_records.append({
        'cmdb_id': cm,
        'kpi_name': kpi,
        'global_p95': gp95,
        'global_p5': gp5,
        'earliest_anom_timestamp_shanghai': earliest_sh,
        'max_value_in_window': max_val,
        'min_value_in_window': min_val,
        'metric_severity_ratio': sev,
        'deviation_pct': deviation
    })

mysql_df = pd.DataFrame(mysql_records)
# filter high-severity: metric_severity_ratio >= 2.0 OR deviation_pct >= 0.5
if not mysql_df.empty:
    mysql_df_filtered = mysql_df[
        (mysql_df['metric_severity_ratio'].fillna(-np.inf) >= 2.0) |
        (mysql_df['deviation_pct'].fillna(-np.inf) >= 0.5)
    ].copy()
    # sort by metric_severity_ratio desc, NaNs to bottom
    mysql_df_filtered['sev_sort'] = mysql_df_filtered['metric_severity_ratio'].fillna(-np.inf)
    mysql_df_filtered = mysql_df_filtered.sort_values('sev_sort', ascending=False).drop(columns=['sev_sort']).head(10).reset_index(drop=True)
    # present selected columns
    mysql_top_high = mysql_df_filtered[[
        'cmdb_id','kpi_name','global_p95','global_p5','earliest_anom_timestamp_shanghai','max_value_in_window','metric_severity_ratio'
    ]].copy()
else:
    mysql_top_high = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','global_p5','earliest_anom_timestamp_shanghai','max_value_in_window','metric_severity_ratio'])

# Final compact outputs:
# - candidate component one-row summary (candidate_summary)
# - per-component compact trace snapshot aggregates (trace_snapshot_aggregates)
# - Mysql02 high-severity KPI list (mysql_top_high)

candidate_summary, trace_snapshot_aggregates, mysql_top_high

```
Out[7]:
```
summary = (
    "Diagnostics summary (candidate true faults):\n\n"
    "1) Priority components (candidate_true_fault == True): MG01, MG02, IG01, IG02, apache01.\n\n"
    "2) Key metric findings:\n"
    "- MG01 & MG02: sustained 31-minute run (2021-03-25 23:00:00 UTC+8 → 23:30:00) on JVM HeapMemoryMax. "
    "Extremal == global_p95, severity ~1.0. This is a long, sustained condition and highest-priority to investigate.\n"
    "- IG01 & IG02: same pattern as MG nodes — 31-minute sustained JVM HeapMemoryMax runs starting 2021-03-25 23:00:00 UTC+8 (severity ~1.0).\n"
    "- apache01: 2-minute high memory (MEMUsedMemPerc) starting 2021-03-25 23:00:00 UTC+8 (severity ~1.0).\n"
    "- Other components (Tomcat nodes, Redis, Mysql02, apache02) show single-minute breaches with small deviations; lower immediate priority.\n\n"
    "3) Trace snapshot aggregates at run start (compact):\n"
    "- MG01: 51 trace rows at run start; top trace names include many 'trace.to_MG01.*'; MG01 appears as downstream callee (other components call MG01).\n"
    "- MG02: 51 trace rows; top 'trace.to_MG02.*'; MG02 is also a downstream callee.\n"
    "- IG01: 27 trace rows; top traces are 'trace.from_IG01.*' (IG01 appears as caller, not downstream callee).\n"
    "- IG02: 27 trace rows; top traces are 'trace.from_IG02.*' (IG02 appears as caller).\n"
    "- apache01: no trace rows at run start in the window (0 rows found).\n\n"
    "4) Mysql02 high-severity KPI check (top candidates found):\n"
    "- Mysql02 shows significant local-disk KPIs later in the window (earliest anomalies at ~2021-03-25 23:09 and 23:09):\n"
    "  * OSLinux...sda_DSKRTps — max ~5.58, metric_severity_ratio ~1.63 (earliest 23:09)\n"
    "  * OSLinux...sda_DSKRead — max ~143.33, metric_severity_ratio ~1.60 (earliest 23:09)\n"
    "  These indicate notable disk read/write activity on Mysql02 after the 23:00 memory/heap incidents.\n\n"
    "5) Short conclusion and recommended priorities:\n"
    "- Investigate MG01/MG02 and IG01/IG02 JVM HeapMemoryMax first (sustained 31-min runs starting 23:00) — they are persistent and likely impactful.\n"
    "- Check apache01 memory spike (2-minute run) as secondary.\n"
    "- Investigate Mysql02 disk read/write anomalies around 23:09 as a potential downstream contributor.\n"
    "- Treat single-minute Tomcat/Redis/apache02 breaches as lower priority but correlate with traces if user-facing errors occur.\n\n"
    "Stored artifacts for follow-up:\n"
    "- component_fault_runs (detailed run metrics)\n"
    "- component_fault_trace_snapshots (full per-component trace snapshots at run starts)\n    - snapshot keys available for: " + ", ".join(sorted(component_fault_trace_snapshots.keys())) + "\n"
    "- metric_thresholds and metric_window (per-KPI thresholds and windowed metric rows)\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  run_start_epoch       run_start_shanghai  run_length_minutes                                  earliest_kpi breach_type  extremal_value_in_run    global_p95     global_p5  deviation_pct  metric_severity_ratio  candidate_true_fault
0      MG01       1616684400  2021-03-25 23:00:00 CST                  31      JVM-Memory_7779_JVM_Memory_HeapMemoryMax        high           2.040136e+09  2.040136e+09  2.040136e+09            0.0                    1.0                  True
1      MG02       1616684400  2021-03-25 23:00:00 CST                  31      JVM-Memory_7779_JVM_Memory_HeapMemoryMax        high           2.040136e+09  2.040136e+09  2.040136e+09            0.0                    1.0                  True
2      IG01       1616684400  2021-03-25 23:00:00 CST                  31      JVM-Memory_7778_JVM_Memory_HeapMemoryMax        high           2.040136e+09  2.040136e+09  2.040136e+09            0.0                    1.0                  True
3      IG02       1616684400  2021-03-25 23:00:00 CST                  31      JVM-Memory_7778_JVM_Memory_HeapMemoryMax        high           2.040136e+09  2.040136e+09  2.040136e+09            0.0                    1.0                  True
4  apache01       1616684400  2021-03-25 23:00:00 CST                   2  OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc        high           7.800000e+01  7.800000e+01  7.700000e+01            0.0                    1.0                  True,     cmdb_id  run_start_epoch       run_start_shanghai  number_of_rows_found                              top_trace_name_counts  is_downstream_callee
0      MG01       1616684400  2021-03-25 23:00:00 CST                    51  trace.to_MG01.duration_p95:4,trace.to_MG01.dur...                  True
1      MG02       1616684400  2021-03-25 23:00:00 CST                    51  trace.to_MG02.duration_p95:4,trace.to_MG02.dur...                  True
2      IG01       1616684400  2021-03-25 23:00:00 CST                    27  trace.from_IG01.duration_mean:4,trace.from_IG0...                 False
3      IG02       1616684400  2021-03-25 23:00:00 CST                    27  trace.from_IG02.duration_mean:4,trace.from_IG0...                 False
4  apache01       1616684400  2021-03-25 23:00:00 CST                     0                                                NaN                 False,    cmdb_id                                         kpi_name    global_p95  global_p5 earliest_anom_timestamp_shanghai  max_value_in_window  metric_severity_ratio
0  Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps  3.423300e+00    0.01332          2021-03-25 23:09:00 CST         5.583300e+00               1.630970
1  Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  8.946666e+01    0.10668          2021-03-25 23:09:00 CST         1.433333e+02               1.602086
2  Mysql02                     Mysql-MySQL_3306_SlowQueries  1.730779e+07    0.72546          2021-03-25 23:21:00 CST         1.730797e+07               0.321589
3  Mysql02                Mysql-MySQL_3306_Handler Rollback  6.676860e+07    2.51192          2021-03-25 23:08:00 CST         6.676860e+07               0.000000
4  Mysql02            Mysql-MySQL_3306_Innodb Row Lock Time  1.404900e+07    0.01356          2021-03-25 23:19:00 CST         1.404903e+07               0.000000
5  Mysql02           Mysql-MySQL_3306_Innodb Row Lock Waits  8.576002e+05    0.01660          2021-03-25 23:19:00 CST         8.576090e+05               0.000000)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse in-kernel variables
for v in ('component_fault_runs','component_fault_trace_snapshots','trace_anomaly_top20','anomaly_filtered'):
    if v not in globals():
        raise NameError(f"Required variable '{v}' not found in kernel.")

tz = pytz.timezone('Asia/Shanghai')

# Candidate components (explicit order)
candidates = ['MG01','MG02','IG01','IG02','apache01','Mysql02','Tomcat02','Tomcat01','Tomcat03']

cfr = component_fault_runs.copy()
snap_map = component_fault_trace_snapshots if isinstance(component_fault_trace_snapshots, dict) else dict(component_fault_trace_snapshots)
tat = trace_anomaly_top20.copy()
af = anomaly_filtered.copy()

# Helper to get earliest metric anomaly epoch for a component
def get_earliest_metric_epoch(cm):
    # prefer component_fault_runs entry
    sub = cfr[cfr['cmdb_id']==cm]
    if not sub.empty:
        # take min run_start_epoch
        return int(sub['run_start_epoch'].min())
    # fallback to anomaly_filtered earliest_anomaly_timestamp
    sub2 = af[af['cmdb_id']==cm]
    if not sub2.empty:
        ts = sub2['earliest_anomaly_timestamp'].dropna()
        if not ts.empty:
            t0 = ts.min()
            # ensure tz-aware and convert to epoch UTC
            try:
                epoch = int(t0.tz_convert(pytz.utc).timestamp())
            except Exception:
                epoch = int(pd.to_datetime(t0).tz_localize(tz).tz_convert(pytz.utc).timestamp())
            return epoch
    return np.nan

# Helper to format epoch to Shanghai string
def epoch_to_shanghai(e):
    if pd.isna(e):
        return np.nan
    ts = pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz)
    return ts.strftime('%Y-%m-%d %H:%M:%S %Z')

# earliest trace anomaly epoch from trace_anomaly_top20
def get_earliest_trace_epoch(cm):
    sub = tat[tat['cmdb_id']==cm]
    if sub.empty:
        return np.nan
    ts = sub['earliest_anomaly_timestamp'].dropna()
    if ts.empty:
        return np.nan
    t0 = ts.min()
    try:
        epoch = int(t0.tz_convert(pytz.utc).timestamp())
    except Exception:
        epoch = int(pd.to_datetime(t0).tz_localize(tz).tz_convert(pytz.utc).timestamp())
    return epoch

# max metric severity ratio from anomaly_filtered
af_sev = af.groupby('cmdb_id', as_index=False).agg(max_metric_severity_ratio=('severity_ratio', lambda s: float(s.max(skipna=True)) if s.notna().any() else np.nan))

rows = []
for cm in candidates:
    # 1-3 earliest metric anomaly
    em_epoch = get_earliest_metric_epoch(cm)
    em_sh = epoch_to_shanghai(em_epoch) if not pd.isna(em_epoch) else np.nan

    # 4 is_downstream_callee_at_earliest: check snapshot at that epoch in snap_map
    is_downstream = False
    if (not pd.isna(em_epoch)) and (cm in snap_map):
        snap = snap_map.get(cm)
        if isinstance(snap, pd.DataFrame) and not snap.empty:
            # look for trace_name containing to_<cm> where caller (cmdb_id) != cm OR any 'to_<cm>' exists
            mask_to = snap['trace_name'].astype(str).str.contains(f"to_{cm}", na=False)
            if mask_to.any():
                # if any of those rows have cmdb_id != cm then other services call this component
                callers = snap.loc[mask_to, 'cmdb_id']
                if ((callers != cm).any()) or True:
                    # mark True if any to_<cm> exists (component is a callee)
                    is_downstream = True

    # 5-6 earliest trace anomaly
    et_epoch = get_earliest_trace_epoch(cm)
    et_sh = epoch_to_shanghai(et_epoch) if not pd.isna(et_epoch) else np.nan

    # 7 max_metric_severity_ratio
    mv = af_sev.loc[af_sev['cmdb_id']==cm, 'max_metric_severity_ratio']
    max_sev = float(mv.iloc[0]) if not mv.empty and pd.notna(mv.iloc[0]) else np.nan

    # 8 candidate_true_fault from component_fault_runs if present
    ctf_row = cfr[cfr['cmdb_id']==cm]
    if not ctf_row.empty:
        ctf_val = bool(ctf_row['candidate_true_fault'].any())
    else:
        ctf_val = False

    # 9 notes: callee / caller / no-trace / large-late-trace
    notes = []
    if is_downstream:
        notes.append('callee')
    else:
        # check if snapshot exists and contains 'from_<cm>' entries => component is caller
        if (cm in snap_map) and isinstance(snap_map.get(cm), pd.DataFrame) and not snap_map.get(cm).empty:
            if snap_map.get(cm)['trace_name'].astype(str).str.contains(f"from_{cm}", na=False).any():
                notes.append('caller')
            else:
                notes.append('no-trace')
        else:
            notes.append('no-trace')
    # large-late-trace: check trace_anomaly_top20 severity for this component >5
    tat_sub = tat[tat['cmdb_id']==cm]
    if not tat_sub.empty:
        if 'severity_ratio' in tat_sub.columns:
            if tat_sub['severity_ratio'].max(skipna=True) > 5:
                notes.append('large-late-trace')
    note_str = ";".join(notes) if notes else ""

    rows.append({
        'cmdb_id': cm,
        'earliest_metric_anom_epoch': int(em_epoch) if not pd.isna(em_epoch) else np.nan,
        'earliest_metric_anom_shanghai': em_sh,
        'is_downstream_callee_at_earliest': bool(is_downstream),
        'earliest_trace_anom_epoch': int(et_epoch) if not pd.isna(et_epoch) else np.nan,
        'earliest_trace_anom_shanghai': et_sh,
        'max_metric_severity_ratio': max_sev,
        'candidate_true_fault': bool(ctf_val),
        'notes': note_str
    })

candidate_ranking = pd.DataFrame(rows)

# Sorting: candidate_true_fault True first, then earliest_metric_anom_epoch asc (NaNs last), then max_metric_severity_ratio desc
candidate_ranking['earliest_sort'] = candidate_ranking['earliest_metric_anom_epoch'].fillna(10**18)
candidate_ranking['sev_sort'] = candidate_ranking['max_metric_severity_ratio'].fillna(-np.inf)
candidate_ranking = candidate_ranking.sort_values(['candidate_true_fault','earliest_sort','sev_sort'], ascending=[False, True, False]).drop(columns=['earliest_sort','sev_sort']).reset_index(drop=True)

# Keep compact and limit to top 20 (we only have 9)
candidate_ranking = candidate_ranking.head(20)

# Final output variable (stored in kernel)
candidate_ranking

```
Out[8]:
```
Summary of ranked candidates (compact):

- MG01
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: True (many incoming "to_MG01" traces)
  - Earliest trace anomaly: 2021-03-25 23:01:00 (UTC+8)
  - Max metric severity (window): ~2.65
  - candidate_true_fault: True
  - Notes: callee; large/late trace activity

- IG01
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False (appears as caller)
  - Earliest trace anomaly: 2021-03-25 23:02:00 (UTC+8)
  - Max metric severity: ~2.57
  - candidate_true_fault: True
  - Notes: caller

- MG02
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: True (many "to_MG02" traces)
  - Earliest trace anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Max metric severity: ~2.27
  - candidate_true_fault: True
  - Notes: callee

- IG02
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False (appears as caller)
  - Earliest trace anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Max metric severity: ~1.76
  - candidate_true_fault: True
  - Notes: caller

- apache01
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False / no trace rows found at that timestamp
  - No trace anomaly in top trace table
  - Max metric severity: ~1.67
  - candidate_true_fault: True
  - Notes: no-trace

- Mysql02
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False / no incoming-trace evidence at that epoch
  - No earliest trace anomaly recorded in top traces
  - Max metric severity: 10.0 (highest in window)
  - candidate_true_fault: False
  - Notes: no-trace — however Mysql02 shows very large metric severity for some KPIs (follow-up required; disk I/O anomalies around ~23:09 seen earlier)

- Tomcat01
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False
  - No earliest trace anomaly in top trace table
  - Max metric severity: ~5.58
  - candidate_true_fault: False
  - Notes: no-trace

- Tomcat03
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False
  - Earliest trace anomaly: 2021-03-25 23:02:00 (UTC+8)
  - Max metric severity: 4.00
  - candidate_true_fault: False
  - Notes: no-trace

- Tomcat02
  - Earliest metric anomaly: 2021-03-25 23:00:00 (UTC+8)
  - Is downstream callee at that time: False
  - Earliest trace anomaly: 2021-03-25 23:01:00 (UTC+8)
  - Max metric severity: ~3.74
  - candidate_true_fault: False
  - Notes: no-trace; large-late-trace

Key takeaways and recommended priorities
- Highest-priority investigation: MG01 and MG02 (sustained JVM HeapMemoryMax runs starting 23:00 and many incoming "to_<MG>" traces) and IG01/IG02 (sustained JVM heap runs but acting as callers). These are candidate true faults (candidate_true_fault == True).
- apache01 also flagged as candidate_true_fault (memory spike) but lacked trace rows at run start.
- Mysql02 shows the largest metric severity (10.0) in the window for some KPIs (not marked candidate_true_fault here). Mysql02 also has notable disk read/write anomalies later (~23:09); investigate as a possible downstream impact or secondary issue.
- Tomcat nodes show high per-KPI severities (Tomcat01/02/03), but their breaches were single-minute (not sustained) — treat as secondary but correlate with traces and user-facing errors if any.

Stored kernel variables for follow-up
- candidate_ranking (this table)
- component_fault_runs, component_fault_trace_snapshots, trace_anomaly_top20, anomaly_filtered, metric_window, metric_thresholds — use these for deeper drill-down and to inspect the saved snapshots and full details.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  earliest_metric_anom_epoch earliest_metric_anom_shanghai  is_downstream_callee_at_earliest  earliest_trace_anom_epoch earliest_trace_anom_shanghai  max_metric_severity_ratio  candidate_true_fault                      notes
0      MG01                  1616684400       2021-03-25 23:00:00 CST                              True               1.616684e+09      2021-03-25 23:01:00 CST                   2.647059                  True    callee;large-late-trace
1      IG01                  1616684400       2021-03-25 23:00:00 CST                             False               1.616685e+09      2021-03-25 23:02:00 CST                   2.570586                  True                     caller
2      MG02                  1616684400       2021-03-25 23:00:00 CST                              True               1.616684e+09      2021-03-25 23:00:00 CST                   2.272727                  True                     callee
3      IG02                  1616684400       2021-03-25 23:00:00 CST                             False               1.616684e+09      2021-03-25 23:00:00 CST                   1.755207                  True                     caller
4  apache01                  1616684400       2021-03-25 23:00:00 CST                             False                        NaN                          NaN                   1.666667                  True                   no-trace
5   Mysql02                  1616684400       2021-03-25 23:00:00 CST                             False                        NaN                          NaN                  10.000000                 False                   no-trace
6  Tomcat01                  1616684400       2021-03-25 23:00:00 CST                             False                        NaN                          NaN                   5.581602                 False                   no-trace
7  Tomcat03                  1616684400       2021-03-25 23:00:00 CST                             False               1.616685e+09      2021-03-25 23:02:00 CST                   4.000000                 False                   no-trace
8  Tomcat02                  1616684400       2021-03-25 23:00:00 CST                             False               1.616684e+09      2021-03-25 23:01:00 CST                   3.743665                 False  no-trace;large-late-trace```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing in-kernel variable: trace_window
if 'trace_window' not in globals():
    raise NameError("trace_window not found in kernel.")

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is tz-aware Asia/Shanghai
if 'timestamp_dt' not in trace_window.columns:
    trace_window['timestamp_dt'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz (if naive, localize then convert)
    if trace_window['timestamp_dt'].dt.tz is None:
        trace_window['timestamp_dt'] = trace_window['timestamp_dt'].dt.tz_localize(pytz.utc).dt.tz_convert(tz)
    else:
        trace_window['timestamp_dt'] = trace_window['timestamp_dt'].dt.tz_convert(tz)

# Target timestamps (UTC+8)
ts_list_sh = ['2021-03-25 23:09:00', '2021-03-25 23:11:00']
ts_list = [pd.to_datetime(s).tz_localize(tz) for s in ts_list_sh]

# Selection criteria
cmdb_whitelist = ['Tomcat02','Mysql02','MG01','MG02','IG01','IG02']
trace_name_patterns = ['to_MG01','to_MG02','to_Mysql02','to_Tomcat02','from_IG01','from_IG02']

# Build mask: timestamp match AND (cmdb_id in list OR trace_name contains any pattern)
mask_time = trace_window['timestamp_dt'].isin(ts_list)
# prepare combined trace_name contains patterns mask
tn_series = trace_window['trace_name'].astype(str)
mask_name = pd.Series(False, index=trace_window.index)
for pat in trace_name_patterns:
    mask_name = mask_name | tn_series.str.contains(pat, na=False)
mask_cmdb = trace_window['cmdb_id'].isin(cmdb_whitelist)

mask_total = mask_time & (mask_cmdb | mask_name)
matched = trace_window[mask_total].copy().reset_index(drop=True)

# Store full matched rows for later inspection
candidate_trace_snapshots = matched.copy()  # kernel variable as requested

# Build aggregated table:
# For grouping label: if trace_name contains 'from_IG01' or 'from_IG02' then label as 'caller', else use cmdb_id value
def label_row(row):
    tn = str(row['trace_name'])
    if re.search(r'\bfrom_IG01\b', tn) or re.search(r'\bfrom_IG02\b', tn):
        return 'caller'
    return row['cmdb_id']

matched['cmdb_label'] = matched.apply(label_row, axis=1)

# Aggregate by timestamp_dt and cmdb_label
def top_trace_counts(series, top_n=5):
    vc = series.value_counts().head(top_n)
    return ",".join([f"{name}:{int(cnt)}" for name,cnt in zip(vc.index, vc.values)])

agg = (matched
       .groupby(['timestamp_dt','cmdb_label'], as_index=False)
       .agg(row_count=('trace_name','size'),
            top_trace_name_counts=('trace_name', lambda s: top_trace_counts(s, top_n=5)),
            indicates_call_to_Mysql02=('trace_name', lambda s: s.str.contains('to_Mysql02', na=False).any()),
            indicates_call_to_MG01orMG02=('trace_name', lambda s: s.str.contains('to_MG01|to_MG02', na=False).any())
           )
      )

# Add human-friendly timestamp_shanghai
agg['timestamp_shanghai'] = agg['timestamp_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Reorder columns per spec and sort by timestamp asc then row_count desc
aggregated_table = agg[[
    'timestamp_shanghai','cmdb_label','row_count','top_trace_name_counts',
    'indicates_call_to_Mysql02','indicates_call_to_MG01orMG02'
]].rename(columns={'cmdb_label':'cmdb_id'})

aggregated_table = aggregated_table.sort_values(['timestamp_shanghai','row_count'], ascending=[True, False]).reset_index(drop=True).head(20)

# Output the compact aggregated table (and candidate_trace_snapshots is stored in kernel)
aggregated_table

```
Out[9]:
```
Summary of trace snapshots at 2021-03-25 23:09:00 and 23:11:00 (UTC+8):

- Snapshot sizes (per timestamp):
  - MG01: 39 rows (both 23:09 and 23:11)
  - MG02: 39 rows (both 23:09 and 23:11)
  - caller (rows with trace_name like from_IG*): 24 rows
  - IG01 / IG02: 15 rows each
  - Tomcat02: 9 rows
  - Tomcat01 / Tomcat03 / Tomcat04: 6 rows each

- Call relationships observed:
  - Many traces are of the form trace.to_MG01 / trace.to_MG02 (Tomcat nodes show these), so MG01 and MG02 are downstream callees at these timestamps.
  - Tomcat nodes (Tomcat02, Tomcat01, Tomcat03, Tomcat04) show trace entries indicating they call MG nodes (aggregates show to_MG01/to_MG02 counts). Tomcat02 also has self.* traces at the same times.
  - IG nodes appear as callers (trace.from_IG01 / trace.from_IG02) in the snapshots — IG → Tomcat → MG is the observed flow.
  - No matched traces at these timestamps indicate calls to Mysql02 (indicates_call_to_Mysql02 = False).

- Interpretation / implication:
  - At both 23:09 and 23:11, MG01 and MG02 are active downstream services receiving many calls from Tomcat nodes; TG (Tomcat) → MG is the prominent path in the snapshots.
  - IG services act as upstream callers into Tomcat at these times.
  - Because there are no 'to_Mysql02' traces at these timestamps, Mysql02’s later disk anomalies (~23:09 observed in metrics) do not appear to be receiving direct trace requests in these snapshots — Mysql02 may be affected later or by different traffic.
  - These snapshots support a topology: IG → Tomcat → MG (MG nodes are downstream callees at the examined anomaly times).

- Data stored for follow-up:
  - Full matched trace rows have been saved in kernel variable candidate_trace_snapshots for detailed inspection.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp_shanghai   cmdb_id  row_count                              top_trace_name_counts  indicates_call_to_Mysql02  indicates_call_to_MG01orMG02
0   2021-03-25 23:09:00 CST      MG01         39  trace.from_Tomcat01.duration_mean:1,trace.from...                      False                         False
1   2021-03-25 23:09:00 CST      MG02         39  trace.from_Tomcat01.duration_mean:1,trace.from...                      False                         False
2   2021-03-25 23:09:00 CST    caller         24  trace.from_IG01.duration_mean:4,trace.from_IG0...                      False                         False
3   2021-03-25 23:09:00 CST      IG01         15  trace.self.duration_mean:1,trace.self.duration...                      False                         False
4   2021-03-25 23:09:00 CST      IG02         15  trace.self.duration_mean:1,trace.self.duration...                      False                         False
5   2021-03-25 23:09:00 CST  Tomcat02          9  trace.self.duration_mean:1,trace.self.duration...                      False                          True
6   2021-03-25 23:09:00 CST  Tomcat01          6  trace.to_MG01.duration_mean:1,trace.to_MG01.du...                      False                          True
7   2021-03-25 23:09:00 CST  Tomcat03          6  trace.to_MG01.duration_mean:1,trace.to_MG01.du...                      False                          True
8   2021-03-25 23:09:00 CST  Tomcat04          6  trace.to_MG01.duration_mean:1,trace.to_MG01.du...                      False                          True
9   2021-03-25 23:11:00 CST      MG01         39  trace.from_Tomcat01.duration_mean:1,trace.from...                      False                         False
10  2021-03-25 23:11:00 CST      MG02         39  trace.from_Tomcat01.duration_mean:1,trace.from...                      False                         False
11  2021-03-25 23:11:00 CST    caller         24  trace.from_IG01.duration_mean:4,trace.from_IG0...                      False                         False
12  2021-03-25 23:11:00 CST      IG01         15  trace.self.duration_mean:1,trace.self.duration...                      False                         False
13  2021-03-25 23:11:00 CST      IG02         15  trace.self.duration_mean:1,trace.self.duration...                      False                         False
14  2021-03-25 23:11:00 CST  Tomcat02          9  trace.self.duration_mean:1,trace.self.duration...                      False                          True
15  2021-03-25 23:11:00 CST  Tomcat01          6  trace.to_MG01.duration_mean:1,trace.to_MG01.du...                      False                          True
16  2021-03-25 23:11:00 CST  Tomcat03          6  trace.to_MG01.duration_mean:1,trace.to_MG01.du...                      False                          True
17  2021-03-25 23:11:00 CST  Tomcat04          6  trace.to_MG01.duration_mean:1,trace.to_MG01.du...                      False                          True```
```